In [ ]:
# ============================================================
# PREPARE SPATIAL FRAGMENTS AND CSV METADATA ONLY
# TABLET-HOLDOUT DATASET
#
# This script:
#   1. Loads manual tablet-holdout annotations.
#   2. Divides each parent tablet image into spatial fragments.
#   3. Saves every extracted spatial fragment as an image.
#   4. Localizes fragments in the parent image.
#   5. Assigns existing annotated sign crops to fragments.
#   6. Saves reusable CSV metadata.
# ============================================================

from __future__ import annotations

import sys
import re
import warnings
from pathlib import Path, PureWindowsPath
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    r"path\cuneiform-ocr-main"
).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.divide_photos import divide_tablet_photo

DATA_ROOT = Path(
    r"path\ebl_tablets_and_sign_crops\sign_classification_extension\ResNet"
)

CSV_PATH = DATA_ROOT / "tablet_holdout_test.csv"

TABLET_IMAGE_DIR = Path(
    r"path\ebl_tablets_and_sign_crops\imgs"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "tablet_holdout_spatial_fragment_dataset"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIVIDED_FRAGMENT_IMAGE_DIR = (
    OUTPUT_DIR / "divided_fragment_images"
)
DIVISION_VISUALIZATION_DIR = (
    OUTPUT_DIR / "division_visualizations"
)

DIVIDED_FRAGMENT_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
DIVISION_VISUALIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Set True only when you intentionally want to rebuild everything.
FORCE_REBUILD = False

USE_TABLET_DIVISION = True
SAVE_DIVIDED_FRAGMENT_IMAGES = True

MIN_FRAGMENT_WIDTH = 40
MIN_FRAGMENT_HEIGHT = 40
MIN_SIGNS_PER_DIVIDED_FRAGMENT = 2
MIN_VALID_FRAGMENTS_PER_QUERY_TABLET = 2
KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS = True

MIN_FRAGMENT_LOCALIZATION_SCORE = 0.55
USE_ORB_FALLBACK = True
MIN_ORB_GOOD_MATCHES = 8
MIN_ORB_INLIERS = 6

MIN_ANNOTATION_OVERLAP_RATIO = 0.50

# Preview generation is intentionally disabled.
CREATE_FRAGMENT_PREVIEWS = False
NUM_TABLETS_TO_PREVIEW = 0

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp",
    ".webp",
}


# ============================================================
# COLUMN ALIASES
# ============================================================

COLUMN_ALIASES: Dict[str, Sequence[str]] = {
    "crop_path": (
        "cropPath",
        "crop_path",
        "sign_crop_path",
    ),
    "tablet_id": (
        "fragmentNumber",
        "tablet_id",
        "fragment_number",
    ),
    "period": (
        "period",
        "true_period",
        "script_period",
    ),
    "sign_name": (
        "signName",
        "label",
        "sign_name",
    ),
    "x": ("x", "x1"),
    "y": ("y", "y1"),
    "width": ("width", "bbox_width"),
    "height": ("height", "bbox_height"),
}

IMAGE_PATH_ALIASES: Sequence[str] = (
    "source_image_path",
    "image_path",
    "imagePath",
    "photo_path",
    "photoPath",
    "tablet_image_path",
    "tabletImagePath",
    "original_image_path",
)

IMAGE_NAME_ALIASES: Sequence[str] = (
    "source_image",
    "image_name",
    "imageName",
    "photo_name",
    "photoName",
    "filename",
)


def safe_name(value: object) -> str:
    """Convert an identifier into a Windows-safe filename component."""
    text = str(value).strip()
    text = re.sub(r'[<>:"/\\|?*]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    return text or "unknown"


def clean_text(series: pd.Series) -> pd.Series:
    """Strip text while preserving missing values."""
    cleaned = series.astype("string").str.strip()
    return cleaned.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "none": pd.NA,
            "null": pd.NA,
            "<na>": pd.NA,
        }
    )


def first_existing_column(
    columns: Iterable[str],
    aliases: Sequence[str],
) -> Optional[str]:
    available = set(columns)
    return next((name for name in aliases if name in available), None)


def load_annotation_dataframe(csv_path: Path) -> pd.DataFrame:
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Tablet-holdout annotation CSV not found: {csv_path}"
        )

    raw_df = pd.read_csv(csv_path, low_memory=False)
    mapping: Dict[str, str] = {}

    for canonical, aliases in COLUMN_ALIASES.items():
        source = first_existing_column(raw_df.columns, aliases)
        if source is None:
            raise ValueError(
                f"Missing required annotation column '{canonical}'.\n"
                f"Accepted aliases: {list(aliases)}\n"
                f"Available columns: {raw_df.columns.tolist()}"
            )
        mapping[canonical] = source

    image_path_source = first_existing_column(
        raw_df.columns,
        IMAGE_PATH_ALIASES,
    )
    image_name_source = first_existing_column(
        raw_df.columns,
        IMAGE_NAME_ALIASES,
    )

    rename_map = {
        source: canonical
        for canonical, source in mapping.items()
        if source != canonical
    }

    df = raw_df.rename(columns=rename_map).copy()

    if image_path_source is not None:
        if image_path_source != "parent_image_path_from_csv":
            df["parent_image_path_from_csv"] = raw_df[image_path_source]

    if image_name_source is not None:
        if image_name_source != "parent_image_name_from_csv":
            df["parent_image_name_from_csv"] = raw_df[image_name_source]

    text_columns = [
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "parent_image_path_from_csv",
        "parent_image_name_from_csv",
    ]
    for column in text_columns:
        if column in df.columns:
            df[column] = clean_text(df[column])

    for column in ["x", "y", "width", "height"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    required = [
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "x",
        "y",
        "width",
        "height",
    ]

    invalid_mask = df[required].isna().any(axis=1)
    invalid_df = df.loc[invalid_mask].copy()
    if not invalid_df.empty:
        invalid_path = OUTPUT_DIR / "invalid_annotation_rows.csv"
        invalid_df.to_csv(invalid_path, index=False)
        print(
            f"Invalid annotation rows excluded: {len(invalid_df):,}\n"
            f"Saved: {invalid_path}"
        )

    df = df.loc[~invalid_mask].copy()
    df = df[(df["width"] > 0) & (df["height"] > 0)].copy()
    df = df.reset_index(drop=True)

    df["source_row_index"] = np.arange(len(df), dtype=np.int64)
    df["x2"] = df["x"] + df["width"]
    df["y2"] = df["y"] + df["height"]
    df["center_x"] = df["x"] + df["width"] / 2.0
    df["center_y"] = df["y"] + df["height"] / 2.0

    print("\nResolved annotation columns:")
    for canonical, source in mapping.items():
        print(f"  {source} -> {canonical}")
    if image_path_source:
        print(f"  {image_path_source} -> parent_image_path_from_csv")
    if image_name_source:
        print(f"  {image_name_source} -> parent_image_name_from_csv")

    print("\nTablet-holdout annotations:")
    print("  Sign annotations:", f"{len(df):,}")
    print("  Parent tablet images:", f"{df['tablet_id'].nunique():,}")

    return df


def build_image_index(root: Path) -> Dict[str, List[Path]]:
    """Index original images by lowercase filename and stem."""
    if not root.is_dir():
        raise FileNotFoundError(
            f"Tablet image directory not found: {root}"
        )

    index: Dict[str, List[Path]] = {}
    paths = [
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for path in paths:
        keys = {
            path.name.lower(),
            path.stem.lower(),
            safe_name(path.stem).lower(),
        }
        for key in keys:
            index.setdefault(key, []).append(path)

    print("Indexed original tablet images:", f"{len(paths):,}")
    return index


def reconstruct_local_path(raw_path: object) -> Optional[Path]:
    if pd.isna(raw_path):
        return None

    text = str(raw_path).strip()
    if not text:
        return None

    direct = Path(text)
    if direct.is_file():
        return direct

    filename = PureWindowsPath(text).name
    if filename:
        candidate = TABLET_IMAGE_DIR / filename
        if candidate.is_file():
            return candidate

    return None


def resolve_parent_image_path(
    tablet_df: pd.DataFrame,
    image_index: Dict[str, List[Path]],
) -> Optional[Path]:
    """Resolve one original image for a parent tablet ID."""
    tablet_id = str(tablet_df["tablet_id"].iloc[0]).strip()

    if "parent_image_path_from_csv" in tablet_df.columns:
        for raw_path in tablet_df["parent_image_path_from_csv"].dropna():
            resolved = reconstruct_local_path(raw_path)
            if resolved is not None:
                return resolved

    candidate_names: List[str] = []

    if "parent_image_name_from_csv" in tablet_df.columns:
        candidate_names.extend(
            str(value).strip()
            for value in tablet_df["parent_image_name_from_csv"].dropna()
        )

    candidate_names.append(tablet_id)

    for candidate_name in candidate_names:
        normalized_name = PureWindowsPath(candidate_name).name
        keys = [
            normalized_name.lower(),
            Path(normalized_name).stem.lower(),
            safe_name(Path(normalized_name).stem).lower(),
        ]

        matches: List[Path] = []
        for key in keys:
            matches.extend(image_index.get(key, []))

        unique_matches = sorted(set(matches))
        if len(unique_matches) == 1:
            return unique_matches[0]

        if len(unique_matches) > 1:
            # Prefer an exact stem match.
            exact = [
                path
                for path in unique_matches
                if path.stem.lower() == Path(normalized_name).stem.lower()
            ]
            if len(exact) == 1:
                return exact[0]

            warnings.warn(
                f"Multiple original images matched tablet {tablet_id}; "
                f"using {unique_matches[0]}"
            )
            return unique_matches[0]

    return None


def validate_divided_fragments(
    regions_bgr: Iterable[np.ndarray],
) -> List[np.ndarray]:
    valid: List[np.ndarray] = []

    for region in regions_bgr:
        if region is None or not isinstance(region, np.ndarray):
            continue
        if region.ndim != 3 or region.shape[2] not in (3, 4):
            continue

        if region.shape[2] == 4:
            region = cv2.cvtColor(region, cv2.COLOR_BGRA2BGR)

        height, width = region.shape[:2]
        if width < MIN_FRAGMENT_WIDTH or height < MIN_FRAGMENT_HEIGHT:
            continue

        valid.append(region)

    return valid


def direct_template_localization(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    """Locate an unchanged divided fragment in its parent photograph."""
    parent_gray = cv2.cvtColor(parent_bgr, cv2.COLOR_BGR2GRAY)
    fragment_gray = cv2.cvtColor(fragment_bgr, cv2.COLOR_BGR2GRAY)

    parent_h, parent_w = parent_gray.shape[:2]
    fragment_h, fragment_w = fragment_gray.shape[:2]

    if fragment_h > parent_h or fragment_w > parent_w:
        return None

    if fragment_h == parent_h and fragment_w == parent_w:
        difference = np.mean(
            np.abs(
                parent_gray.astype(np.float32)
                - fragment_gray.astype(np.float32)
            )
        )
        score = float(max(0.0, 1.0 - difference / 255.0))
        return {
            "x1": 0,
            "y1": 0,
            "x2": parent_w,
            "y2": parent_h,
            "score": score,
            "method": "full_image",
        }

    candidates: List[Tuple[float, Tuple[int, int], str]] = []

    # Intensity correlation.
    if float(fragment_gray.std()) > 1e-6:
        result = cv2.matchTemplate(
            parent_gray,
            fragment_gray,
            cv2.TM_CCOEFF_NORMED,
        )
        _, max_score, _, max_location = cv2.minMaxLoc(result)
        if np.isfinite(max_score):
            candidates.append(
                (float(max_score), max_location, "template_intensity")
            )

    # Normalized squared-difference converted so larger is better.
    result_sq = cv2.matchTemplate(
        parent_gray,
        fragment_gray,
        cv2.TM_SQDIFF_NORMED,
    )
    min_score, _, min_location, _ = cv2.minMaxLoc(result_sq)
    if np.isfinite(min_score):
        candidates.append(
            (float(1.0 - min_score), min_location, "template_sqdiff")
        )

    # Edge correlation is useful when a divided fragment contains a flat or
    # masked background around the tablet material.
    parent_edges = cv2.Canny(parent_gray, 50, 150)
    fragment_edges = cv2.Canny(fragment_gray, 50, 150)
    if np.count_nonzero(fragment_edges) >= 20:
        edge_result = cv2.matchTemplate(
            parent_edges,
            fragment_edges,
            cv2.TM_CCOEFF_NORMED,
        )
        _, edge_score, _, edge_location = cv2.minMaxLoc(edge_result)
        if np.isfinite(edge_score):
            candidates.append(
                (float(edge_score), edge_location, "template_edges")
            )

    if not candidates:
        return None

    best_score, (x1, y1), method = max(candidates, key=lambda item: item[0])

    return {
        "x1": int(x1),
        "y1": int(y1),
        "x2": int(x1 + fragment_w),
        "y2": int(y1 + fragment_h),
        "score": float(best_score),
        "method": method,
    }


def orb_fragment_localization(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    """ORB/homography fallback when direct template matching is weak."""
    parent_gray = cv2.cvtColor(parent_bgr, cv2.COLOR_BGR2GRAY)
    fragment_gray = cv2.cvtColor(fragment_bgr, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create(nfeatures=5000)
    fragment_keypoints, fragment_descriptors = orb.detectAndCompute(
        fragment_gray,
        None,
    )
    parent_keypoints, parent_descriptors = orb.detectAndCompute(
        parent_gray,
        None,
    )

    if (
        fragment_descriptors is None
        or parent_descriptors is None
        or len(fragment_keypoints) < MIN_ORB_GOOD_MATCHES
        or len(parent_keypoints) < MIN_ORB_GOOD_MATCHES
    ):
        return None

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    knn_matches = matcher.knnMatch(
        fragment_descriptors,
        parent_descriptors,
        k=2,
    )

    good_matches = []
    for pair in knn_matches:
        if len(pair) != 2:
            continue
        first, second = pair
        if first.distance < 0.75 * second.distance:
            good_matches.append(first)

    if len(good_matches) < MIN_ORB_GOOD_MATCHES:
        return None

    source_points = np.float32(
        [fragment_keypoints[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)
    destination_points = np.float32(
        [parent_keypoints[m.trainIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    homography, inlier_mask = cv2.findHomography(
        source_points,
        destination_points,
        cv2.RANSAC,
        4.0,
    )

    if homography is None or inlier_mask is None:
        return None

    num_inliers = int(inlier_mask.ravel().sum())
    if num_inliers < MIN_ORB_INLIERS:
        return None

    fragment_h, fragment_w = fragment_gray.shape[:2]
    corners = np.float32(
        [
            [0, 0],
            [fragment_w - 1, 0],
            [fragment_w - 1, fragment_h - 1],
            [0, fragment_h - 1],
        ]
    ).reshape(-1, 1, 2)

    projected = cv2.perspectiveTransform(corners, homography).reshape(-1, 2)

    parent_h, parent_w = parent_gray.shape[:2]
    x1 = int(np.floor(projected[:, 0].min()))
    y1 = int(np.floor(projected[:, 1].min()))
    x2 = int(np.ceil(projected[:, 0].max())) + 1
    y2 = int(np.ceil(projected[:, 1].max())) + 1

    x1 = max(0, min(x1, parent_w - 1))
    y1 = max(0, min(y1, parent_h - 1))
    x2 = max(x1 + 1, min(x2, parent_w))
    y2 = max(y1 + 1, min(y2, parent_h))

    inlier_ratio = num_inliers / max(len(good_matches), 1)

    return {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "score": float(inlier_ratio),
        "method": "orb_homography",
        "orb_good_matches": int(len(good_matches)),
        "orb_inliers": num_inliers,
    }


def localize_fragment_in_parent(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    direct = direct_template_localization(parent_bgr, fragment_bgr)

    if (
        direct is not None
        and float(direct["score"]) >= MIN_FRAGMENT_LOCALIZATION_SCORE
    ):
        return direct

    if USE_ORB_FALLBACK:
        orb_result = orb_fragment_localization(parent_bgr, fragment_bgr)
        if orb_result is not None:
            return orb_result

    return direct


def divide_and_localize_tablet(
    tablet_id: str,
    image_path: Path,
) -> Tuple[np.ndarray, List[np.ndarray], List[Dict[str, object]]]:
    parent_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if parent_bgr is None:
        raise RuntimeError(f"OpenCV could not read: {image_path}")

    tablet_safe_name = safe_name(tablet_id)
    visualization_path = (
        DIVISION_VISUALIZATION_DIR
        / f"{tablet_safe_name}_division_visualization.jpg"
    )

    if USE_TABLET_DIVISION:
        regions = divide_tablet_photo(
            str(image_path),
            visualize=False,
            output_path=str(visualization_path),
        )
        if regions is None:
            raise RuntimeError(
                f"divide_tablet_photo returned None for {image_path}"
            )
        regions_bgr = validate_divided_fragments(list(regions))
    else:
        regions_bgr = [parent_bgr]

    if not regions_bgr:
        raise RuntimeError(
            f"No valid divided fragments were returned for {image_path}"
        )

    localized_records: List[Dict[str, object]] = []

    tablet_fragment_dir = (
        DIVIDED_FRAGMENT_IMAGE_DIR / tablet_safe_name
    )
    if SAVE_DIVIDED_FRAGMENT_IMAGES:
        tablet_fragment_dir.mkdir(parents=True, exist_ok=True)

    for fragment_index, region_bgr in enumerate(regions_bgr):
        localization = localize_fragment_in_parent(
            parent_bgr,
            region_bgr,
        )

        fragment_id = f"F{fragment_index:03d}"
        fragment_uid = f"{tablet_id}_divided_fragment_{fragment_index:03d}"

        fragment_path = ""
        if SAVE_DIVIDED_FRAGMENT_IMAGES:
            saved_path = tablet_fragment_dir / f"{fragment_id}.png"
            if not cv2.imwrite(str(saved_path), region_bgr):
                warnings.warn(f"Could not save divided fragment: {saved_path}")
            else:
                fragment_path = str(saved_path)

        if localization is None:
            localized_records.append(
                {
                    "tablet_id": tablet_id,
                    "parent_image_path": str(image_path),
                    "fragment_id": fragment_id,
                    "fragment_uid": fragment_uid,
                    "fragment_path": fragment_path,
                    "fragment_width_pixels": int(region_bgr.shape[1]),
                    "fragment_height_pixels": int(region_bgr.shape[0]),
                    "localization_status": "failed",
                    "localization_method": "",
                    "localization_score": np.nan,
                    "fragment_x1": np.nan,
                    "fragment_y1": np.nan,
                    "fragment_x2": np.nan,
                    "fragment_y2": np.nan,
                }
            )
            continue

        localization_score = float(localization["score"])
        status = (
            "localized"
            if localization["method"] == "orb_homography"
            or localization_score >= MIN_FRAGMENT_LOCALIZATION_SCORE
            else "weak_localization"
        )

        localized_records.append(
            {
                "tablet_id": tablet_id,
                "parent_image_path": str(image_path),
                "fragment_id": fragment_id,
                "fragment_uid": fragment_uid,
                "fragment_path": fragment_path,
                "fragment_width_pixels": int(region_bgr.shape[1]),
                "fragment_height_pixels": int(region_bgr.shape[0]),
                "localization_status": status,
                "localization_method": localization["method"],
                "localization_score": localization_score,
                "fragment_x1": int(localization["x1"]),
                "fragment_y1": int(localization["y1"]),
                "fragment_x2": int(localization["x2"]),
                "fragment_y2": int(localization["y2"]),
                "orb_good_matches": localization.get(
                    "orb_good_matches",
                    np.nan,
                ),
                "orb_inliers": localization.get("orb_inliers", np.nan),
            }
        )

    return parent_bgr, regions_bgr, localized_records


def annotation_overlap_ratio(
    annotation_box: Tuple[float, float, float, float],
    fragment_box: Tuple[float, float, float, float],
) -> float:
    ax1, ay1, ax2, ay2 = annotation_box
    fx1, fy1, fx2, fy2 = fragment_box

    intersection_x1 = max(ax1, fx1)
    intersection_y1 = max(ay1, fy1)
    intersection_x2 = min(ax2, fx2)
    intersection_y2 = min(ay2, fy2)

    intersection_width = max(0.0, intersection_x2 - intersection_x1)
    intersection_height = max(0.0, intersection_y2 - intersection_y1)
    intersection_area = intersection_width * intersection_height

    annotation_area = max((ax2 - ax1) * (ay2 - ay1), 1e-8)
    return float(intersection_area / annotation_area)


def assign_tablet_annotations(
    tablet_df: pd.DataFrame,
    fragment_records: List[Dict[str, object]],
) -> List[Dict[str, object]]:
    assignments: List[Dict[str, object]] = []

    usable_fragments = [
        record
        for record in fragment_records
        if record["localization_status"] == "localized"
    ]

    for _, row in tablet_df.iterrows():
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        annotation_box = (
            float(row["x"]),
            float(row["y"]),
            float(row["x2"]),
            float(row["y2"]),
        )

        center_candidates: List[Tuple[float, Dict[str, object]]] = []

        for fragment in usable_fragments:
            fx1 = float(fragment["fragment_x1"])
            fy1 = float(fragment["fragment_y1"])
            fx2 = float(fragment["fragment_x2"])
            fy2 = float(fragment["fragment_y2"])

            if fx1 <= center_x < fx2 and fy1 <= center_y < fy2:
                overlap = annotation_overlap_ratio(
                    annotation_box,
                    (fx1, fy1, fx2, fy2),
                )
                center_candidates.append((overlap, fragment))

        assignment_method = ""
        best_overlap = 0.0
        selected_fragment: Optional[Dict[str, object]] = None

        if center_candidates:
            best_overlap, selected_fragment = max(
                center_candidates,
                key=lambda item: (
                    item[0],
                    float(item[1]["localization_score"]),
                    -(
                        (float(item[1]["fragment_x2"]) - float(item[1]["fragment_x1"]))
                        * (float(item[1]["fragment_y2"]) - float(item[1]["fragment_y1"]))
                    ),
                ),
            )
            assignment_method = "annotation_center"

        else:
            overlap_candidates: List[Tuple[float, Dict[str, object]]] = []

            for fragment in usable_fragments:
                fragment_box = (
                    float(fragment["fragment_x1"]),
                    float(fragment["fragment_y1"]),
                    float(fragment["fragment_x2"]),
                    float(fragment["fragment_y2"]),
                )
                overlap = annotation_overlap_ratio(
                    annotation_box,
                    fragment_box,
                )
                overlap_candidates.append((overlap, fragment))

            if overlap_candidates:
                best_overlap, best_fragment = max(
                    overlap_candidates,
                    key=lambda item: (
                        item[0],
                        float(item[1]["localization_score"]),
                    ),
                )
                if best_overlap >= MIN_ANNOTATION_OVERLAP_RATIO:
                    selected_fragment = best_fragment
                    assignment_method = "annotation_overlap"

        base_record = {
            "source_row_index": int(row["source_row_index"]),
            "tablet_id": str(row["tablet_id"]),
            "crop_path": str(row["crop_path"]),
            "sign_name": str(row["sign_name"]),
            "period": str(row["period"]),
            "annotation_x": float(row["x"]),
            "annotation_y": float(row["y"]),
            "annotation_x2": float(row["x2"]),
            "annotation_y2": float(row["y2"]),
            "annotation_center_x": center_x,
            "annotation_center_y": center_y,
            "assignment_overlap_ratio": float(best_overlap),
        }

        if selected_fragment is None:
            base_record.update(
                {
                    "assignment_status": "unassigned",
                    "assignment_method": "",
                    "fragment_id": "",
                    "fragment_uid": "",
                    "fragment_path": "",
                    "fragment_x1": np.nan,
                    "fragment_y1": np.nan,
                    "fragment_x2": np.nan,
                    "fragment_y2": np.nan,
                }
            )
        else:
            base_record.update(
                {
                    "assignment_status": "assigned",
                    "assignment_method": assignment_method,
                    "fragment_id": selected_fragment["fragment_id"],
                    "fragment_uid": selected_fragment["fragment_uid"],
                    "fragment_path": selected_fragment["fragment_path"],
                    "fragment_x1": selected_fragment["fragment_x1"],
                    "fragment_y1": selected_fragment["fragment_y1"],
                    "fragment_x2": selected_fragment["fragment_x2"],
                    "fragment_y2": selected_fragment["fragment_y2"],
                }
            )

        assignments.append(base_record)

    return assignments


def prepare_divided_fragments_and_assignments(
    dataframe: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide every parent image once and assign existing annotations."""
    image_index = build_image_index(TABLET_IMAGE_DIR)

    fragment_records_all: List[Dict[str, object]] = []
    assignment_records_all: List[Dict[str, object]] = []
    tablet_status_rows: List[Dict[str, object]] = []

    grouped = dataframe.groupby("tablet_id", sort=False)

    for tablet_id, tablet_df in tqdm(
        grouped,
        total=dataframe["tablet_id"].nunique(),
        desc="Dividing tablet images and assigning annotations",
    ):
        tablet_id = str(tablet_id)
        image_path = resolve_parent_image_path(tablet_df, image_index)

        if image_path is None:
            tablet_status_rows.append(
                {
                    "tablet_id": tablet_id,
                    "status": "parent_image_not_found",
                    "parent_image_path": "",
                    "num_annotations": int(len(tablet_df)),
                    "num_divided_fragments": 0,
                    "num_localized_fragments": 0,
                    "num_assigned_annotations": 0,
                }
            )
            continue

        try:
            (
                parent_bgr,
                regions_bgr,
                fragment_records,
            ) = divide_and_localize_tablet(
                tablet_id=tablet_id,
                image_path=image_path,
            )
        except Exception as error:
            tablet_status_rows.append(
                {
                    "tablet_id": tablet_id,
                    "status": "division_or_localization_error",
                    "parent_image_path": str(image_path),
                    "error": str(error),
                    "num_annotations": int(len(tablet_df)),
                    "num_divided_fragments": 0,
                    "num_localized_fragments": 0,
                    "num_assigned_annotations": 0,
                }
            )
            continue

        assignment_records = assign_tablet_annotations(
            tablet_df,
            fragment_records,
        )
        assignments_tablet_df = pd.DataFrame(assignment_records)

        fragment_records_all.extend(fragment_records)
        assignment_records_all.extend(assignment_records)

        num_localized = sum(
            record["localization_status"] == "localized"
            for record in fragment_records
        )
        num_assigned = int(
            (
                assignments_tablet_df["assignment_status"]
                == "assigned"
            ).sum()
        )

        tablet_status_rows.append(
            {
                "tablet_id": tablet_id,
                "status": "processed",
                "parent_image_path": str(image_path),
                "num_annotations": int(len(tablet_df)),
                "num_divided_fragments": int(len(fragment_records)),
                "num_localized_fragments": int(num_localized),
                "num_assigned_annotations": num_assigned,
                "num_unassigned_annotations": int(len(tablet_df) - num_assigned),
            }
        )


    fragment_df = pd.DataFrame(fragment_records_all)
    assignments_df = pd.DataFrame(assignment_records_all)
    tablet_status_df = pd.DataFrame(tablet_status_rows)

    if fragment_df.empty:
        raise ValueError(
            "No divided fragments were generated from the tablet-holdout images."
        )

    if assignments_df.empty:
        raise ValueError(
            "No annotation assignments were generated."
        )

    fragment_df.to_csv(
        OUTPUT_DIR / "all_divided_fragment_localizations.csv",
        index=False,
    )
    assignments_df.to_csv(
        OUTPUT_DIR / "all_annotation_to_fragment_assignments.csv",
        index=False,
    )
    tablet_status_df.to_csv(
        OUTPUT_DIR / "tablet_division_processing_status.csv",
        index=False,
    )

    return fragment_df, assignments_df, tablet_status_df


def build_valid_fragment_dataset(
    annotation_df: pd.DataFrame,
    localization_df: pd.DataFrame,
    assignments_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    assigned = assignments_df[
        assignments_df["assignment_status"] == "assigned"
    ].copy()

    assigned_counts = (
        assigned.groupby("fragment_uid")
        .size()
        .rename("num_signs")
        .reset_index()
    )

    valid_localizations = localization_df[
        localization_df["localization_status"] == "localized"
    ].copy()

    fragment_df = valid_localizations.merge(
        assigned_counts,
        on="fragment_uid",
        how="left",
    )
    fragment_df["num_signs"] = fragment_df["num_signs"].fillna(0).astype(int)

    fragment_df = fragment_df[
        fragment_df["num_signs"] >= MIN_SIGNS_PER_DIVIDED_FRAGMENT
    ].copy()

    if fragment_df.empty:
        raise ValueError(
            "No divided fragments contain the required minimum number of "
            f"assigned signs ({MIN_SIGNS_PER_DIVIDED_FRAGMENT})."
        )

    fragment_counts = (
        fragment_df.groupby("tablet_id")
        .size()
        .rename("num_valid_fragments_in_tablet")
    )

    fragment_df = fragment_df.merge(
        fragment_counts,
        left_on="tablet_id",
        right_index=True,
        how="left",
    )

    if not KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS:
        keep_tablets = fragment_counts[
            fragment_counts >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET
        ].index
        fragment_df = fragment_df[
            fragment_df["tablet_id"].isin(keep_tablets)
        ].copy()

    valid_fragment_uids = set(fragment_df["fragment_uid"])
    assigned = assigned[
        assigned["fragment_uid"].isin(valid_fragment_uids)
    ].copy()

    # Join the original annotation rows using the stable source-row index.
    annotation_columns = [
        "source_row_index",
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "x",
        "y",
        "x2",
        "y2",
        "center_x",
        "center_y",
    ]

    annotation_lookup = annotation_df[annotation_columns].copy()

    assigned = assigned.drop(
        columns=[
            column
            for column in [
                "crop_path",
                "tablet_id",
                "period",
                "sign_name",
            ]
            if column in assigned.columns
        ]
    )

    assigned_sign_df = assigned.merge(
        annotation_lookup,
        on="source_row_index",
        how="inner",
    )

    assigned_sign_df = assigned_sign_df.sort_values(
        ["tablet_id", "fragment_uid", "source_row_index"]
    ).reset_index(drop=True)

    fragment_period = (
        assigned_sign_df.groupby("fragment_uid")["period"]
        .agg(lambda values: values.mode().iloc[0])
        .rename("period")
        .reset_index()
    )

    fragment_sign_names = (
        assigned_sign_df.groupby("fragment_uid")["sign_name"]
        .agg(lambda values: ";".join(map(str, values)))
        .rename("sign_names")
        .reset_index()
    )

    source_indices = (
        assigned_sign_df.groupby("fragment_uid")["source_row_index"]
        .agg(lambda values: ";".join(map(str, values.astype(int))))
        .rename("source_row_indices")
        .reset_index()
    )

    fragment_df = fragment_df.merge(
        fragment_period,
        on="fragment_uid",
        how="left",
    )
    fragment_df = fragment_df.merge(
        fragment_sign_names,
        on="fragment_uid",
        how="left",
    )
    fragment_df = fragment_df.merge(
        source_indices,
        on="fragment_uid",
        how="left",
    )

    fragment_df = fragment_df.sort_values(
        ["tablet_id", "fragment_id"]
    ).reset_index(drop=True)

    fragment_df.to_csv(
        OUTPUT_DIR / "valid_divided_fragment_metadata.csv",
        index=False,
    )
    assigned_sign_df.to_csv(
        OUTPUT_DIR / "assigned_signs_used_for_embeddings.csv",
        index=False,
    )

    print("\nDIVIDED-FRAGMENT DATASET")
    print("------------------------")
    print("Assigned signs used:", f"{len(assigned_sign_df):,}")
    print("Valid divided fragments:", f"{len(fragment_df):,}")
    print(
        "Parent tablets represented:",
        f"{fragment_df['tablet_id'].nunique():,}",
    )
    print(
        "Tablets with at least two valid fragments:",
        f"{fragment_df.loc[fragment_df['num_valid_fragments_in_tablet'] >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET, 'tablet_id'].nunique():,}",
    )

    return fragment_df, assigned_sign_df



# ============================================================
# MODEL-INDEPENDENT MANIFESTS
# ============================================================

def build_model_independent_manifests(
    valid_fragment_df: pd.DataFrame,
    assigned_sign_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build stable manifests that can be consumed unchanged by:
    ResNet, ConvNeXt, ViT, Swin, or another image encoder.

    The preparation stage never applies a model-specific transform.
    Each downstream model reads crop_path, extracts one sign embedding
    per row, and aggregates rows sharing the same fragment_uid.
    """

    required_fragment_columns = {
        "tablet_id",
        "fragment_id",
        "fragment_uid",
        "fragment_path",
        "parent_image_path",
        "num_signs",
    }

    required_sign_columns = {
        "source_row_index",
        "tablet_id",
        "fragment_id",
        "fragment_uid",
        "fragment_path",
        "crop_path",
        "sign_name",
        "period",
    }

    missing_fragment_columns = (
        required_fragment_columns - set(valid_fragment_df.columns)
    )
    missing_sign_columns = (
        required_sign_columns - set(assigned_sign_df.columns)
    )

    if missing_fragment_columns:
        raise ValueError(
            "Valid-fragment metadata is missing required columns: "
            f"{sorted(missing_fragment_columns)}"
        )

    if missing_sign_columns:
        raise ValueError(
            "Assigned-sign metadata is missing required columns: "
            f"{sorted(missing_sign_columns)}"
        )

    fragment_manifest = valid_fragment_df.copy()
    sign_manifest = assigned_sign_df.copy()

    # Canonical text formatting prevents model blocks from grouping the same
    # identifier differently because of whitespace or mixed dtypes.
    for dataframe, columns in (
        (
            fragment_manifest,
            [
                "tablet_id",
                "fragment_id",
                "fragment_uid",
                "fragment_path",
                "parent_image_path",
                "period",
            ],
        ),
        (
            sign_manifest,
            [
                "tablet_id",
                "fragment_id",
                "fragment_uid",
                "fragment_path",
                "crop_path",
                "sign_name",
                "period",
            ],
        ),
    ):
        for column in columns:
            if column in dataframe.columns:
                dataframe[column] = (
                    dataframe[column]
                    .astype("string")
                    .str.strip()
                )

    # Use a deterministic order. The embedding produced for row i must always
    # correspond to sign_manifest row i for every architecture.
    sign_manifest = sign_manifest.sort_values(
        ["tablet_id", "fragment_uid", "source_row_index"],
        kind="stable",
    ).reset_index(drop=True)

    fragment_manifest = fragment_manifest.sort_values(
        ["tablet_id", "fragment_id", "fragment_uid"],
        kind="stable",
    ).reset_index(drop=True)

    sign_manifest.insert(
        0,
        "embedding_row_index",
        np.arange(len(sign_manifest), dtype=np.int64),
    )

    fragment_manifest.insert(
        0,
        "fragment_row_index",
        np.arange(len(fragment_manifest), dtype=np.int64),
    )

    # Validate one-to-one fragment identifiers.
    if fragment_manifest["fragment_uid"].duplicated().any():
        duplicates = (
            fragment_manifest.loc[
                fragment_manifest["fragment_uid"].duplicated(keep=False),
                "fragment_uid",
            ]
            .astype(str)
            .unique()
            .tolist()
        )
        raise ValueError(
            "fragment_uid must be unique in the fragment manifest. "
            f"Examples: {duplicates[:10]}"
        )

    # Every assigned sign must point to a valid fragment.
    valid_fragment_uids = set(
        fragment_manifest["fragment_uid"].dropna().astype(str)
    )
    sign_fragment_uids = set(
        sign_manifest["fragment_uid"].dropna().astype(str)
    )

    missing_fragment_links = sign_fragment_uids - valid_fragment_uids
    if missing_fragment_links:
        raise ValueError(
            "Assigned signs reference fragment_uid values that are absent "
            "from the valid-fragment manifest. Examples: "
            f"{sorted(missing_fragment_links)[:10]}"
        )

    # The extracted fragment files are required for future experiments that
    # encode the whole spatial fragment directly.
    missing_fragment_files = [
        path
        for path in fragment_manifest["fragment_path"].dropna().astype(str)
        if path.strip() and not Path(path).is_file()
    ]

    if missing_fragment_files:
        warnings.warn(
            f"{len(missing_fragment_files):,} fragment image paths do not "
            "currently exist. The CSVs remain usable for sign-crop aggregation, "
            "but direct fragment-image encoding will fail."
        )

    # Sign crop paths are required by all current model-specific retrieval blocks.
    missing_crop_files = [
        path
        for path in sign_manifest["crop_path"].dropna().astype(str)
        if path.strip() and not Path(path).is_file()
    ]

    if missing_crop_files:
        warnings.warn(
            f"{len(missing_crop_files):,} sign crop paths do not currently "
            "exist. Downstream model extraction must recover or correct them."
        )

    return fragment_manifest, sign_manifest


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    """
    Prepare and save the model-independent spatial-fragment dataset.

    Run this block once. All later ResNet, ConvNeXt, ViT, Swin, or other
    retrieval blocks should only read the generated files.
    """

    final_files = {
        "all_localizations": (
            OUTPUT_DIR / "all_divided_fragment_localizations.csv"
        ),
        "all_assignments": (
            OUTPUT_DIR / "all_annotation_to_fragment_assignments.csv"
        ),
        "tablet_status": (
            OUTPUT_DIR / "tablet_division_processing_status.csv"
        ),
        "valid_fragments": (
            OUTPUT_DIR / "valid_divided_fragment_metadata.csv"
        ),
        "assigned_signs": (
            OUTPUT_DIR / "assigned_signs_used_for_embeddings.csv"
        ),
        "fragment_manifest": (
            OUTPUT_DIR / "model_input_fragment_manifest.csv"
        ),
        "sign_manifest": (
            OUTPUT_DIR / "model_input_sign_manifest.csv"
        ),
        "summary": (
            OUTPUT_DIR / "spatial_fragment_dataset_summary.csv"
        ),
    }

    print("Annotation CSV:", CSV_PATH)
    print("Tablet image directory:", TABLET_IMAGE_DIR)
    print("Output directory:", OUTPUT_DIR)
    print("Spatial-fragment image directory:", DIVIDED_FRAGMENT_IMAGE_DIR)

    all_outputs_exist = all(
        path.is_file() for path in final_files.values()
    )

    if all_outputs_exist and not FORCE_REBUILD:
        print("\nDATASET ALREADY PREPARED")
        print("------------------------")
        print("No tablet images will be divided again.")

        fragment_manifest = pd.read_csv(
            final_files["fragment_manifest"],
            low_memory=False,
        )
        sign_manifest = pd.read_csv(
            final_files["sign_manifest"],
            low_memory=False,
        )

        print("Valid fragments:", f"{len(fragment_manifest):,}")
        print("Assigned signs:", f"{len(sign_manifest):,}")
        print(
            "Parent tablets:",
            f"{fragment_manifest['tablet_id'].nunique():,}",
        )

        print("\nReusable model inputs:")
        print(final_files["fragment_manifest"])
        print(final_files["sign_manifest"])
        print(DIVIDED_FRAGMENT_IMAGE_DIR)
        print(
            "\nSet FORCE_REBUILD=True only when you intentionally "
            "want to regenerate the spatial-fragment dataset."
        )
        return

    annotation_df = load_annotation_dataframe(CSV_PATH)

    (
        localization_df,
        assignments_df,
        tablet_status_df,
    ) = prepare_divided_fragments_and_assignments(
        annotation_df
    )

    (
        valid_fragment_df,
        assigned_sign_df,
    ) = build_valid_fragment_dataset(
        annotation_df=annotation_df,
        localization_df=localization_df,
        assignments_df=assignments_df,
    )

    fragment_manifest, sign_manifest = (
        build_model_independent_manifests(
            valid_fragment_df=valid_fragment_df,
            assigned_sign_df=assigned_sign_df,
        )
    )

    # Save the canonical model-independent tables under both the historical
    # filenames and explicit model-input filenames. Existing retrieval code
    # can continue reading assigned_signs_used_for_embeddings.csv and
    # valid_divided_fragment_metadata.csv without modification.
    fragment_manifest.to_csv(
        final_files["valid_fragments"],
        index=False,
    )
    sign_manifest.to_csv(
        final_files["assigned_signs"],
        index=False,
    )

    fragment_manifest.to_csv(
        final_files["fragment_manifest"],
        index=False,
    )
    sign_manifest.to_csv(
        final_files["sign_manifest"],
        index=False,
    )

    num_multi_fragment_tablets = int(
        fragment_manifest.loc[
            fragment_manifest["num_valid_fragments_in_tablet"]
            >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET,
            "tablet_id",
        ].nunique()
    )

    summary_df = pd.DataFrame(
        [
            {
                "dataset_type": "model_independent_spatial_fragments",
                "compatible_model_families": (
                    "ResNet;ConvNeXt;ViT;Swin;other_image_encoders"
                ),
                "num_annotation_rows": len(annotation_df),
                "num_parent_tablets_in_annotations": (
                    annotation_df["tablet_id"].nunique()
                ),
                "num_all_divided_fragments": len(localization_df),
                "num_localized_fragments": int(
                    (
                        localization_df["localization_status"]
                        == "localized"
                    ).sum()
                ),
                "num_valid_fragments": len(fragment_manifest),
                "num_assigned_signs": len(sign_manifest),
                "num_parent_tablets_with_valid_fragments": (
                    fragment_manifest["tablet_id"].nunique()
                ),
                "num_parent_tablets_with_multiple_fragments": (
                    num_multi_fragment_tablets
                ),
                "minimum_signs_per_valid_fragment": (
                    MIN_SIGNS_PER_DIVIDED_FRAGMENT
                ),
                "fragment_image_root": str(
                    DIVIDED_FRAGMENT_IMAGE_DIR
                ),
                "fragment_manifest": str(
                    final_files["fragment_manifest"]
                ),
                "sign_manifest": str(
                    final_files["sign_manifest"]
                ),
            }
        ]
    )

    summary_df.to_csv(
        final_files["summary"],
        index=False,
    )

    print("\n" + "=" * 80)
    print("MODEL-INDEPENDENT SPATIAL-FRAGMENT DATASET READY")
    print("=" * 80)
    print("All divided fragments:", f"{len(localization_df):,}")
    print("Valid fragments:", f"{len(fragment_manifest):,}")
    print("Assigned sign crops:", f"{len(sign_manifest):,}")
    print(
        "Parent tablets represented:",
        f"{fragment_manifest['tablet_id'].nunique():,}",
    )
    print(
        "Tablets with multiple valid fragments:",
        f"{num_multi_fragment_tablets:,}",
    )

    print("\nFiles for any deep model:")
    print(final_files["fragment_manifest"])
    print(final_files["sign_manifest"])
    print(DIVIDED_FRAGMENT_IMAGE_DIR)

    print("\nCompatibility aliases:")
    print(final_files["valid_fragments"])
    print(final_files["assigned_signs"])

    print(
        "\nNo checkpoint was loaded. No model transform, embedding, "
        "shuffling, similarity matrix, or retrieval evaluation was run."
    )


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# IMAGE-DIVISION FRAGMENT RETRIEVAL
# CONVNEXT BASE — TABLET-HOLDOUT SETTING
# ============================================================
# Pipeline
# --------
# 1. Load the manually annotated tablet-holdout CSV.
# 2. Load each parent tablet photograph.
# 3. Divide the photograph with divide_tablet_photo().
# 4. Recover the location of each returned image fragment in the parent image.
# 5. Assign existing sign annotations to divided fragments using annotation boxes.
# 6. Extract one ConvNeXt embedding per existing annotated sign crop.
# 7. Average sign embeddings inside each divided image fragment.
# 8. Pool all valid divided fragments and retrieve another fragment from the
#    same parent tablet image at ranks 1, 5, and 10.
# ============================================================

from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import warnings
from pathlib import Path, PureWindowsPath
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

import os
import sys
import gc
import json
import random

#from data_processing.divide_photos import divide_tablet_photo


# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    r"path\cuneiform-ocr-main"
).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.divide_photos import divide_tablet_photo

DATA_ROOT = Path(
    r"path\ebl_tablets_and_sign_crops\sign_classification_extension\ResNet"
)

CSV_PATH = DATA_ROOT / "tablet_holdout_test.csv"

# Folder containing the original/full tablet photographs.
# The script searches this folder recursively by fragmentNumber and by any
# image-path column present in the CSV.
TABLET_IMAGE_DIR = Path(
    r"path\ebl_tablets_and_sign_crops\imgs"
)

# Search all sign-classification experiment folders for the ConvNeXt checkpoint.
MODEL_SEARCH_ROOT = DATA_ROOT.parent

# The spatial-fragment metadata was already produced by the ResNet run.
# It is reused here so tablet images are never divided again.
FRAGMENT_METADATA_DIR = (
    PROJECT_ROOT
    / "tablet_holdout_spatial_fragment_dataset"#"tablet_holdout_divided_fragment_retrieval_resnet"
)

MODEL_SPECS = {
    "convnext_base_tablet_holdout": {
        "architecture": "convnext_base",
        "checkpoint_candidates": (
            "best_convnext_base_tablet_holdout_sign_classifier.pth",
            "best_convnext_tablet_holdout_sign_classifier.pth",
            "convnext_base_tablet_holdout_sign_classifier.pth",
            "convnext_tablet_holdout_sign_classifier.pth",
            "best_convnext_base_tablet_holdout.pth",
            "best_convnext_tablet_holdout.pth",
        ),
    },
}


OUTPUT_DIR = (
    PROJECT_ROOT
    / "tablet_holdout_divided_fragment_retrieval_convnext"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These paths are retained only for compatibility with helper functions.
# main() never invokes tablet division.
DIVIDED_FRAGMENT_IMAGE_DIR = (
    FRAGMENT_METADATA_DIR / "divided_fragment_images"
)
DIVISION_VISUALIZATION_DIR = (
    FRAGMENT_METADATA_DIR / "division_visualizations"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = DEVICE == "cuda"
RANDOM_SEED = 42

# Fragment division and assignment settings.
USE_TABLET_DIVISION = True
MIN_FRAGMENT_WIDTH = 40
MIN_FRAGMENT_HEIGHT = 40
MIN_SIGNS_PER_DIVIDED_FRAGMENT = 2
MIN_VALID_FRAGMENTS_PER_QUERY_TABLET = 2
KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS = True

# Localization settings. Direct template matching normally reaches a score
# close to 1.0 when divide_tablet_photo returns an unchanged crop.
MIN_FRAGMENT_LOCALIZATION_SCORE = 0.55
USE_ORB_FALLBACK = True
MIN_ORB_GOOD_MATCHES = 8
MIN_ORB_INLIERS = 6

# Annotation assignment.
# First use annotation-box centers. If a center does not fall inside any
# localized fragment, use annotation-box overlap as a fallback.
MIN_ANNOTATION_OVERLAP_RATIO = 0.50

# Preview and saved fragment settings.
SAVE_DIVIDED_FRAGMENT_IMAGES = False
CREATE_FRAGMENT_PREVIEWS = False
PREVIEW_ONLY = False
NUM_TABLETS_TO_PREVIEW = 15

# Retrieval settings.
TOP_K_TO_SAVE = 10
SHUFFLE_FRAGMENT_GALLERY = True

# Image extensions used for original-image indexing.
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("Using device:", DEVICE)
print("Annotation CSV:", CSV_PATH)
print("Tablet image directory:", TABLET_IMAGE_DIR)
print("Output directory:", OUTPUT_DIR)


# ============================================================
# GENERAL HELPERS
# ============================================================

def safe_name(value: object) -> str:
    """Convert an identifier into a Windows-safe filename component."""
    text = str(value).strip()
    text = re.sub(r'[<>:"/\\|?*]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    return text or "unknown"


def clean_text(series: pd.Series) -> pd.Series:
    """Strip text while preserving missing values."""
    cleaned = series.astype("string").str.strip()
    return cleaned.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "none": pd.NA,
            "null": pd.NA,
            "<na>": pd.NA,
        }
    )


def first_existing_column(
    columns: Iterable[str],
    aliases: Sequence[str],
) -> Optional[str]:
    available = set(columns)
    return next((name for name in aliases if name in available), None)


def first_valid(values: Iterable[object]) -> object:
    for value in values:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return ""


def display_or_print(dataframe: pd.DataFrame, n: int = 30) -> None:
    preview = dataframe.head(n)
    try:
        from IPython.display import display
        display(preview)
    except ImportError:
        print(preview.to_string(index=False))


# ============================================================
# ANNOTATION CSV COLUMN RESOLUTION
# ============================================================

COLUMN_ALIASES: Dict[str, Sequence[str]] = {
    "crop_path": (
        "cropPath",
        "crop_path",
        "sign_crop_path",
    ),
    "tablet_id": (
        "fragmentNumber",
        "tablet_id",
        "fragment_number",
    ),
    "period": (
        "period",
        "true_period",
        "script_period",
    ),
    "sign_name": (
        "signName",
        "label",
        "sign_name",
    ),
    "x": (
        "x",
        "x1",
    ),
    "y": (
        "y",
        "y1",
    ),
    "width": (
        "width",
        "bbox_width",
    ),
    "height": (
        "height",
        "bbox_height",
    ),
}

IMAGE_PATH_ALIASES: Sequence[str] = (
    "source_image_path",
    "image_path",
    "imagePath",
    "photo_path",
    "photoPath",
    "tablet_image_path",
    "tabletImagePath",
    "original_image_path",
)

IMAGE_NAME_ALIASES: Sequence[str] = (
    "source_image",
    "image_name",
    "imageName",
    "photo_name",
    "photoName",
    "filename",
)


def load_annotation_dataframe(csv_path: Path) -> pd.DataFrame:
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Tablet-holdout annotation CSV not found: {csv_path}"
        )

    raw_df = pd.read_csv(csv_path, low_memory=False)
    mapping: Dict[str, str] = {}

    for canonical, aliases in COLUMN_ALIASES.items():
        source = first_existing_column(raw_df.columns, aliases)
        if source is None:
            raise ValueError(
                f"Missing required annotation column '{canonical}'.\n"
                f"Accepted aliases: {list(aliases)}\n"
                f"Available columns: {raw_df.columns.tolist()}"
            )
        mapping[canonical] = source

    image_path_source = first_existing_column(
        raw_df.columns,
        IMAGE_PATH_ALIASES,
    )
    image_name_source = first_existing_column(
        raw_df.columns,
        IMAGE_NAME_ALIASES,
    )

    rename_map = {
        source: canonical
        for canonical, source in mapping.items()
        if source != canonical
    }

    df = raw_df.rename(columns=rename_map).copy()

    if image_path_source is not None:
        if image_path_source != "parent_image_path_from_csv":
            df["parent_image_path_from_csv"] = raw_df[image_path_source]

    if image_name_source is not None:
        if image_name_source != "parent_image_name_from_csv":
            df["parent_image_name_from_csv"] = raw_df[image_name_source]

    text_columns = [
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "parent_image_path_from_csv",
        "parent_image_name_from_csv",
    ]
    for column in text_columns:
        if column in df.columns:
            df[column] = clean_text(df[column])

    for column in ["x", "y", "width", "height"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    required = [
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "x",
        "y",
        "width",
        "height",
    ]

    invalid_mask = df[required].isna().any(axis=1)
    invalid_df = df.loc[invalid_mask].copy()
    if not invalid_df.empty:
        invalid_path = OUTPUT_DIR / "invalid_annotation_rows.csv"
        invalid_df.to_csv(invalid_path, index=False)
        print(
            f"Invalid annotation rows excluded: {len(invalid_df):,}\n"
            f"Saved: {invalid_path}"
        )

    df = df.loc[~invalid_mask].copy()
    df = df[(df["width"] > 0) & (df["height"] > 0)].copy()
    df = df.reset_index(drop=True)

    df["source_row_index"] = np.arange(len(df), dtype=np.int64)
    df["x2"] = df["x"] + df["width"]
    df["y2"] = df["y"] + df["height"]
    df["center_x"] = df["x"] + df["width"] / 2.0
    df["center_y"] = df["y"] + df["height"] / 2.0

    print("\nResolved annotation columns:")
    for canonical, source in mapping.items():
        print(f"  {source} -> {canonical}")
    if image_path_source:
        print(f"  {image_path_source} -> parent_image_path_from_csv")
    if image_name_source:
        print(f"  {image_name_source} -> parent_image_name_from_csv")

    print("\nTablet-holdout annotations:")
    print("  Sign annotations:", f"{len(df):,}")
    print("  Parent tablet images:", f"{df['tablet_id'].nunique():,}")

    return df


# ============================================================
# ORIGINAL TABLET-IMAGE RESOLUTION
# ============================================================

def build_image_index(root: Path) -> Dict[str, List[Path]]:
    """Index original images by lowercase filename and stem."""
    if not root.is_dir():
        raise FileNotFoundError(
            f"Tablet image directory not found: {root}"
        )

    index: Dict[str, List[Path]] = {}
    paths = [
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for path in paths:
        keys = {
            path.name.lower(),
            path.stem.lower(),
            safe_name(path.stem).lower(),
        }
        for key in keys:
            index.setdefault(key, []).append(path)

    print("Indexed original tablet images:", f"{len(paths):,}")
    return index


def reconstruct_local_path(raw_path: object) -> Optional[Path]:
    if pd.isna(raw_path):
        return None

    text = str(raw_path).strip()
    if not text:
        return None

    direct = Path(text)
    if direct.is_file():
        return direct

    filename = PureWindowsPath(text).name
    if filename:
        candidate = TABLET_IMAGE_DIR / filename
        if candidate.is_file():
            return candidate

    return None


def resolve_parent_image_path(
    tablet_df: pd.DataFrame,
    image_index: Dict[str, List[Path]],
) -> Optional[Path]:
    """Resolve one original image for a parent tablet ID."""
    tablet_id = str(tablet_df["tablet_id"].iloc[0]).strip()

    if "parent_image_path_from_csv" in tablet_df.columns:
        for raw_path in tablet_df["parent_image_path_from_csv"].dropna():
            resolved = reconstruct_local_path(raw_path)
            if resolved is not None:
                return resolved

    candidate_names: List[str] = []

    if "parent_image_name_from_csv" in tablet_df.columns:
        candidate_names.extend(
            str(value).strip()
            for value in tablet_df["parent_image_name_from_csv"].dropna()
        )

    candidate_names.append(tablet_id)

    for candidate_name in candidate_names:
        normalized_name = PureWindowsPath(candidate_name).name
        keys = [
            normalized_name.lower(),
            Path(normalized_name).stem.lower(),
            safe_name(Path(normalized_name).stem).lower(),
        ]

        matches: List[Path] = []
        for key in keys:
            matches.extend(image_index.get(key, []))

        unique_matches = sorted(set(matches))
        if len(unique_matches) == 1:
            return unique_matches[0]

        if len(unique_matches) > 1:
            # Prefer an exact stem match.
            exact = [
                path
                for path in unique_matches
                if path.stem.lower() == Path(normalized_name).stem.lower()
            ]
            if len(exact) == 1:
                return exact[0]

            warnings.warn(
                f"Multiple original images matched tablet {tablet_id}; "
                f"using {unique_matches[0]}"
            )
            return unique_matches[0]

    return None


# ============================================================
# DIVIDE TABLET IMAGE AND LOCALIZE RETURNED FRAGMENTS
# ============================================================

def validate_divided_fragments(
    regions_bgr: Iterable[np.ndarray],
) -> List[np.ndarray]:
    valid: List[np.ndarray] = []

    for region in regions_bgr:
        if region is None or not isinstance(region, np.ndarray):
            continue
        if region.ndim != 3 or region.shape[2] not in (3, 4):
            continue

        if region.shape[2] == 4:
            region = cv2.cvtColor(region, cv2.COLOR_BGRA2BGR)

        height, width = region.shape[:2]
        if width < MIN_FRAGMENT_WIDTH or height < MIN_FRAGMENT_HEIGHT:
            continue

        valid.append(region)

    return valid


def direct_template_localization(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    """Locate an unchanged divided fragment in its parent photograph."""
    parent_gray = cv2.cvtColor(parent_bgr, cv2.COLOR_BGR2GRAY)
    fragment_gray = cv2.cvtColor(fragment_bgr, cv2.COLOR_BGR2GRAY)

    parent_h, parent_w = parent_gray.shape[:2]
    fragment_h, fragment_w = fragment_gray.shape[:2]

    if fragment_h > parent_h or fragment_w > parent_w:
        return None

    if fragment_h == parent_h and fragment_w == parent_w:
        difference = np.mean(
            np.abs(
                parent_gray.astype(np.float32)
                - fragment_gray.astype(np.float32)
            )
        )
        score = float(max(0.0, 1.0 - difference / 255.0))
        return {
            "x1": 0,
            "y1": 0,
            "x2": parent_w,
            "y2": parent_h,
            "score": score,
            "method": "full_image",
        }

    candidates: List[Tuple[float, Tuple[int, int], str]] = []

    # Intensity correlation.
    if float(fragment_gray.std()) > 1e-6:
        result = cv2.matchTemplate(
            parent_gray,
            fragment_gray,
            cv2.TM_CCOEFF_NORMED,
        )
        _, max_score, _, max_location = cv2.minMaxLoc(result)
        if np.isfinite(max_score):
            candidates.append(
                (float(max_score), max_location, "template_intensity")
            )

    # Normalized squared-difference converted so larger is better.
    result_sq = cv2.matchTemplate(
        parent_gray,
        fragment_gray,
        cv2.TM_SQDIFF_NORMED,
    )
    min_score, _, min_location, _ = cv2.minMaxLoc(result_sq)
    if np.isfinite(min_score):
        candidates.append(
            (float(1.0 - min_score), min_location, "template_sqdiff")
        )

    # Edge correlation is useful when a divided fragment contains a flat or
    # masked background around the tablet material.
    parent_edges = cv2.Canny(parent_gray, 50, 150)
    fragment_edges = cv2.Canny(fragment_gray, 50, 150)
    if np.count_nonzero(fragment_edges) >= 20:
        edge_result = cv2.matchTemplate(
            parent_edges,
            fragment_edges,
            cv2.TM_CCOEFF_NORMED,
        )
        _, edge_score, _, edge_location = cv2.minMaxLoc(edge_result)
        if np.isfinite(edge_score):
            candidates.append(
                (float(edge_score), edge_location, "template_edges")
            )

    if not candidates:
        return None

    best_score, (x1, y1), method = max(candidates, key=lambda item: item[0])

    return {
        "x1": int(x1),
        "y1": int(y1),
        "x2": int(x1 + fragment_w),
        "y2": int(y1 + fragment_h),
        "score": float(best_score),
        "method": method,
    }


def orb_fragment_localization(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    """ORB/homography fallback when direct template matching is weak."""
    parent_gray = cv2.cvtColor(parent_bgr, cv2.COLOR_BGR2GRAY)
    fragment_gray = cv2.cvtColor(fragment_bgr, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create(nfeatures=5000)
    fragment_keypoints, fragment_descriptors = orb.detectAndCompute(
        fragment_gray,
        None,
    )
    parent_keypoints, parent_descriptors = orb.detectAndCompute(
        parent_gray,
        None,
    )

    if (
        fragment_descriptors is None
        or parent_descriptors is None
        or len(fragment_keypoints) < MIN_ORB_GOOD_MATCHES
        or len(parent_keypoints) < MIN_ORB_GOOD_MATCHES
    ):
        return None

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    knn_matches = matcher.knnMatch(
        fragment_descriptors,
        parent_descriptors,
        k=2,
    )

    good_matches = []
    for pair in knn_matches:
        if len(pair) != 2:
            continue
        first, second = pair
        if first.distance < 0.75 * second.distance:
            good_matches.append(first)

    if len(good_matches) < MIN_ORB_GOOD_MATCHES:
        return None

    source_points = np.float32(
        [fragment_keypoints[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)
    destination_points = np.float32(
        [parent_keypoints[m.trainIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    homography, inlier_mask = cv2.findHomography(
        source_points,
        destination_points,
        cv2.RANSAC,
        4.0,
    )

    if homography is None or inlier_mask is None:
        return None

    num_inliers = int(inlier_mask.ravel().sum())
    if num_inliers < MIN_ORB_INLIERS:
        return None

    fragment_h, fragment_w = fragment_gray.shape[:2]
    corners = np.float32(
        [
            [0, 0],
            [fragment_w - 1, 0],
            [fragment_w - 1, fragment_h - 1],
            [0, fragment_h - 1],
        ]
    ).reshape(-1, 1, 2)

    projected = cv2.perspectiveTransform(corners, homography).reshape(-1, 2)

    parent_h, parent_w = parent_gray.shape[:2]
    x1 = int(np.floor(projected[:, 0].min()))
    y1 = int(np.floor(projected[:, 1].min()))
    x2 = int(np.ceil(projected[:, 0].max())) + 1
    y2 = int(np.ceil(projected[:, 1].max())) + 1

    x1 = max(0, min(x1, parent_w - 1))
    y1 = max(0, min(y1, parent_h - 1))
    x2 = max(x1 + 1, min(x2, parent_w))
    y2 = max(y1 + 1, min(y2, parent_h))

    inlier_ratio = num_inliers / max(len(good_matches), 1)

    return {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "score": float(inlier_ratio),
        "method": "orb_homography",
        "orb_good_matches": int(len(good_matches)),
        "orb_inliers": num_inliers,
    }


def localize_fragment_in_parent(
    parent_bgr: np.ndarray,
    fragment_bgr: np.ndarray,
) -> Optional[Dict[str, object]]:
    direct = direct_template_localization(parent_bgr, fragment_bgr)

    if (
        direct is not None
        and float(direct["score"]) >= MIN_FRAGMENT_LOCALIZATION_SCORE
    ):
        return direct

    if USE_ORB_FALLBACK:
        orb_result = orb_fragment_localization(parent_bgr, fragment_bgr)
        if orb_result is not None:
            return orb_result

    return direct


def divide_and_localize_tablet(
    tablet_id: str,
    image_path: Path,
) -> Tuple[np.ndarray, List[np.ndarray], List[Dict[str, object]]]:
    parent_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if parent_bgr is None:
        raise RuntimeError(f"OpenCV could not read: {image_path}")

    tablet_safe_name = safe_name(tablet_id)
    visualization_path = (
        DIVISION_VISUALIZATION_DIR
        / f"{tablet_safe_name}_division_visualization.jpg"
    )

    if USE_TABLET_DIVISION:
        regions = divide_tablet_photo(
            str(image_path),
            visualize=False,
            output_path=str(visualization_path),
        )
        if regions is None:
            raise RuntimeError(
                f"divide_tablet_photo returned None for {image_path}"
            )
        regions_bgr = validate_divided_fragments(list(regions))
    else:
        regions_bgr = [parent_bgr]

    if not regions_bgr:
        raise RuntimeError(
            f"No valid divided fragments were returned for {image_path}"
        )

    localized_records: List[Dict[str, object]] = []

    tablet_fragment_dir = (
        DIVIDED_FRAGMENT_IMAGE_DIR / tablet_safe_name
    )
    if SAVE_DIVIDED_FRAGMENT_IMAGES:
        tablet_fragment_dir.mkdir(parents=True, exist_ok=True)

    for fragment_index, region_bgr in enumerate(regions_bgr):
        localization = localize_fragment_in_parent(
            parent_bgr,
            region_bgr,
        )

        fragment_id = f"F{fragment_index:03d}"
        fragment_uid = f"{tablet_id}_divided_fragment_{fragment_index:03d}"

        fragment_path = ""
        if SAVE_DIVIDED_FRAGMENT_IMAGES:
            saved_path = tablet_fragment_dir / f"{fragment_id}.png"
            if not cv2.imwrite(str(saved_path), region_bgr):
                warnings.warn(f"Could not save divided fragment: {saved_path}")
            else:
                fragment_path = str(saved_path)

        if localization is None:
            localized_records.append(
                {
                    "tablet_id": tablet_id,
                    "parent_image_path": str(image_path),
                    "fragment_id": fragment_id,
                    "fragment_uid": fragment_uid,
                    "fragment_path": fragment_path,
                    "fragment_width_pixels": int(region_bgr.shape[1]),
                    "fragment_height_pixels": int(region_bgr.shape[0]),
                    "localization_status": "failed",
                    "localization_method": "",
                    "localization_score": np.nan,
                    "fragment_x1": np.nan,
                    "fragment_y1": np.nan,
                    "fragment_x2": np.nan,
                    "fragment_y2": np.nan,
                }
            )
            continue

        localization_score = float(localization["score"])
        status = (
            "localized"
            if localization["method"] == "orb_homography"
            or localization_score >= MIN_FRAGMENT_LOCALIZATION_SCORE
            else "weak_localization"
        )

        localized_records.append(
            {
                "tablet_id": tablet_id,
                "parent_image_path": str(image_path),
                "fragment_id": fragment_id,
                "fragment_uid": fragment_uid,
                "fragment_path": fragment_path,
                "fragment_width_pixels": int(region_bgr.shape[1]),
                "fragment_height_pixels": int(region_bgr.shape[0]),
                "localization_status": status,
                "localization_method": localization["method"],
                "localization_score": localization_score,
                "fragment_x1": int(localization["x1"]),
                "fragment_y1": int(localization["y1"]),
                "fragment_x2": int(localization["x2"]),
                "fragment_y2": int(localization["y2"]),
                "orb_good_matches": localization.get(
                    "orb_good_matches",
                    np.nan,
                ),
                "orb_inliers": localization.get("orb_inliers", np.nan),
            }
        )

    return parent_bgr, regions_bgr, localized_records


# ============================================================
# ASSIGN MANUAL ANNOTATIONS TO DIVIDED FRAGMENTS
# ============================================================

def annotation_overlap_ratio(
    annotation_box: Tuple[float, float, float, float],
    fragment_box: Tuple[float, float, float, float],
) -> float:
    ax1, ay1, ax2, ay2 = annotation_box
    fx1, fy1, fx2, fy2 = fragment_box

    intersection_x1 = max(ax1, fx1)
    intersection_y1 = max(ay1, fy1)
    intersection_x2 = min(ax2, fx2)
    intersection_y2 = min(ay2, fy2)

    intersection_width = max(0.0, intersection_x2 - intersection_x1)
    intersection_height = max(0.0, intersection_y2 - intersection_y1)
    intersection_area = intersection_width * intersection_height

    annotation_area = max((ax2 - ax1) * (ay2 - ay1), 1e-8)
    return float(intersection_area / annotation_area)


def assign_tablet_annotations(
    tablet_df: pd.DataFrame,
    fragment_records: List[Dict[str, object]],
) -> List[Dict[str, object]]:
    assignments: List[Dict[str, object]] = []

    usable_fragments = [
        record
        for record in fragment_records
        if record["localization_status"] == "localized"
    ]

    for _, row in tablet_df.iterrows():
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        annotation_box = (
            float(row["x"]),
            float(row["y"]),
            float(row["x2"]),
            float(row["y2"]),
        )

        center_candidates: List[Tuple[float, Dict[str, object]]] = []

        for fragment in usable_fragments:
            fx1 = float(fragment["fragment_x1"])
            fy1 = float(fragment["fragment_y1"])
            fx2 = float(fragment["fragment_x2"])
            fy2 = float(fragment["fragment_y2"])

            if fx1 <= center_x < fx2 and fy1 <= center_y < fy2:
                overlap = annotation_overlap_ratio(
                    annotation_box,
                    (fx1, fy1, fx2, fy2),
                )
                center_candidates.append((overlap, fragment))

        assignment_method = ""
        best_overlap = 0.0
        selected_fragment: Optional[Dict[str, object]] = None

        if center_candidates:
            best_overlap, selected_fragment = max(
                center_candidates,
                key=lambda item: (
                    item[0],
                    float(item[1]["localization_score"]),
                    -(
                        (float(item[1]["fragment_x2"]) - float(item[1]["fragment_x1"]))
                        * (float(item[1]["fragment_y2"]) - float(item[1]["fragment_y1"]))
                    ),
                ),
            )
            assignment_method = "annotation_center"

        else:
            overlap_candidates: List[Tuple[float, Dict[str, object]]] = []

            for fragment in usable_fragments:
                fragment_box = (
                    float(fragment["fragment_x1"]),
                    float(fragment["fragment_y1"]),
                    float(fragment["fragment_x2"]),
                    float(fragment["fragment_y2"]),
                )
                overlap = annotation_overlap_ratio(
                    annotation_box,
                    fragment_box,
                )
                overlap_candidates.append((overlap, fragment))

            if overlap_candidates:
                best_overlap, best_fragment = max(
                    overlap_candidates,
                    key=lambda item: (
                        item[0],
                        float(item[1]["localization_score"]),
                    ),
                )
                if best_overlap >= MIN_ANNOTATION_OVERLAP_RATIO:
                    selected_fragment = best_fragment
                    assignment_method = "annotation_overlap"

        base_record = {
            "source_row_index": int(row["source_row_index"]),
            "tablet_id": str(row["tablet_id"]),
            "crop_path": str(row["crop_path"]),
            "sign_name": str(row["sign_name"]),
            "period": str(row["period"]),
            "annotation_x": float(row["x"]),
            "annotation_y": float(row["y"]),
            "annotation_x2": float(row["x2"]),
            "annotation_y2": float(row["y2"]),
            "annotation_center_x": center_x,
            "annotation_center_y": center_y,
            "assignment_overlap_ratio": float(best_overlap),
        }

        if selected_fragment is None:
            base_record.update(
                {
                    "assignment_status": "unassigned",
                    "assignment_method": "",
                    "fragment_id": "",
                    "fragment_uid": "",
                    "fragment_path": "",
                    "fragment_x1": np.nan,
                    "fragment_y1": np.nan,
                    "fragment_x2": np.nan,
                    "fragment_y2": np.nan,
                }
            )
        else:
            base_record.update(
                {
                    "assignment_status": "assigned",
                    "assignment_method": assignment_method,
                    "fragment_id": selected_fragment["fragment_id"],
                    "fragment_uid": selected_fragment["fragment_uid"],
                    "fragment_path": selected_fragment["fragment_path"],
                    "fragment_x1": selected_fragment["fragment_x1"],
                    "fragment_y1": selected_fragment["fragment_y1"],
                    "fragment_x2": selected_fragment["fragment_x2"],
                    "fragment_y2": selected_fragment["fragment_y2"],
                }
            )

        assignments.append(base_record)

    return assignments


def preview_tablet_division(
    tablet_id: str,
    parent_bgr: np.ndarray,
    regions_bgr: List[np.ndarray],
    fragment_records: List[Dict[str, object]],
    assignments_df: pd.DataFrame,
) -> None:
    """Display the original image with localized fragment boxes and fragments."""
    localized = [
        record
        for record in fragment_records
        if record["localization_status"] == "localized"
    ]

    columns = max(2, min(4, len(regions_bgr) + 1))
    total_panels = len(regions_bgr) + 1
    rows = math.ceil(total_panels / columns)

    figure = plt.figure(figsize=(5 * columns, 5 * rows))

    axis = figure.add_subplot(rows, columns, 1)
    axis.imshow(cv2.cvtColor(parent_bgr, cv2.COLOR_BGR2RGB))
    axis.set_title(f"Parent tablet: {tablet_id}")
    axis.axis("off")

    for record in localized:
        x1 = int(record["fragment_x1"])
        y1 = int(record["fragment_y1"])
        x2 = int(record["fragment_x2"])
        y2 = int(record["fragment_y2"])

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2,
        )
        axis.add_patch(rectangle)

        assigned_count = int(
            (
                assignments_df["fragment_uid"]
                == record["fragment_uid"]
            ).sum()
        )
        axis.text(
            x1,
            y1,
            f"{record['fragment_id']} | signs={assigned_count}",
            fontsize=9,
        )

    for fragment_index, region_bgr in enumerate(regions_bgr):
        panel = figure.add_subplot(rows, columns, fragment_index + 2)
        panel.imshow(cv2.cvtColor(region_bgr, cv2.COLOR_BGR2RGB))

        record = fragment_records[fragment_index]
        assigned_count = int(
            (
                assignments_df["fragment_uid"]
                == record["fragment_uid"]
            ).sum()
        )
        panel.set_title(
            f"{record['fragment_id']} | assigned signs={assigned_count}\n"
            f"{record['localization_method']} | "
            f"score={record['localization_score']:.3f}"
            if pd.notna(record["localization_score"])
            else f"{record['fragment_id']} | localization failed"
        )
        panel.axis("off")

    figure.tight_layout()
    plt.show()
    plt.close(figure)


def prepare_divided_fragments_and_assignments(
    dataframe: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide every parent image once and assign existing annotations."""
    image_index = build_image_index(TABLET_IMAGE_DIR)

    fragment_records_all: List[Dict[str, object]] = []
    assignment_records_all: List[Dict[str, object]] = []
    tablet_status_rows: List[Dict[str, object]] = []

    previewed = 0

    grouped = dataframe.groupby("tablet_id", sort=False)

    for tablet_id, tablet_df in tqdm(
        grouped,
        total=dataframe["tablet_id"].nunique(),
        desc="Dividing tablet images and assigning annotations",
    ):
        tablet_id = str(tablet_id)
        image_path = resolve_parent_image_path(tablet_df, image_index)

        if image_path is None:
            tablet_status_rows.append(
                {
                    "tablet_id": tablet_id,
                    "status": "parent_image_not_found",
                    "parent_image_path": "",
                    "num_annotations": int(len(tablet_df)),
                    "num_divided_fragments": 0,
                    "num_localized_fragments": 0,
                    "num_assigned_annotations": 0,
                }
            )
            continue

        try:
            (
                parent_bgr,
                regions_bgr,
                fragment_records,
            ) = divide_and_localize_tablet(
                tablet_id=tablet_id,
                image_path=image_path,
            )
        except Exception as error:
            tablet_status_rows.append(
                {
                    "tablet_id": tablet_id,
                    "status": "division_or_localization_error",
                    "parent_image_path": str(image_path),
                    "error": str(error),
                    "num_annotations": int(len(tablet_df)),
                    "num_divided_fragments": 0,
                    "num_localized_fragments": 0,
                    "num_assigned_annotations": 0,
                }
            )
            continue

        assignment_records = assign_tablet_annotations(
            tablet_df,
            fragment_records,
        )
        assignments_tablet_df = pd.DataFrame(assignment_records)

        fragment_records_all.extend(fragment_records)
        assignment_records_all.extend(assignment_records)

        num_localized = sum(
            record["localization_status"] == "localized"
            for record in fragment_records
        )
        num_assigned = int(
            (
                assignments_tablet_df["assignment_status"]
                == "assigned"
            ).sum()
        )

        tablet_status_rows.append(
            {
                "tablet_id": tablet_id,
                "status": "processed",
                "parent_image_path": str(image_path),
                "num_annotations": int(len(tablet_df)),
                "num_divided_fragments": int(len(fragment_records)),
                "num_localized_fragments": int(num_localized),
                "num_assigned_annotations": num_assigned,
                "num_unassigned_annotations": int(len(tablet_df) - num_assigned),
            }
        )

        should_preview = (
            CREATE_FRAGMENT_PREVIEWS
            and (
                NUM_TABLETS_TO_PREVIEW is None
                or previewed < NUM_TABLETS_TO_PREVIEW
            )
        )

        if should_preview:
            preview_tablet_division(
                tablet_id=tablet_id,
                parent_bgr=parent_bgr,
                regions_bgr=regions_bgr,
                fragment_records=fragment_records,
                assignments_df=assignments_tablet_df,
            )
            previewed += 1

    fragment_df = pd.DataFrame(fragment_records_all)
    assignments_df = pd.DataFrame(assignment_records_all)
    tablet_status_df = pd.DataFrame(tablet_status_rows)

    if fragment_df.empty:
        raise ValueError(
            "No divided fragments were generated from the tablet-holdout images."
        )

    if assignments_df.empty:
        raise ValueError(
            "No annotation assignments were generated."
        )

    fragment_df.to_csv(
        OUTPUT_DIR / "all_divided_fragment_localizations.csv",
        index=False,
    )
    assignments_df.to_csv(
        OUTPUT_DIR / "all_annotation_to_fragment_assignments.csv",
        index=False,
    )
    tablet_status_df.to_csv(
        OUTPUT_DIR / "tablet_division_processing_status.csv",
        index=False,
    )

    return fragment_df, assignments_df, tablet_status_df


# ============================================================
# BUILD VALID FRAGMENT AND SIGN TABLES
# ============================================================

def build_valid_fragment_dataset(
    annotation_df: pd.DataFrame,
    localization_df: pd.DataFrame,
    assignments_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    assigned = assignments_df[
        assignments_df["assignment_status"] == "assigned"
    ].copy()

    assigned_counts = (
        assigned.groupby("fragment_uid")
        .size()
        .rename("num_signs")
        .reset_index()
    )

    valid_localizations = localization_df[
        localization_df["localization_status"] == "localized"
    ].copy()

    fragment_df = valid_localizations.merge(
        assigned_counts,
        on="fragment_uid",
        how="left",
    )
    fragment_df["num_signs"] = fragment_df["num_signs"].fillna(0).astype(int)

    fragment_df = fragment_df[
        fragment_df["num_signs"] >= MIN_SIGNS_PER_DIVIDED_FRAGMENT
    ].copy()

    if fragment_df.empty:
        raise ValueError(
            "No divided fragments contain the required minimum number of "
            f"assigned signs ({MIN_SIGNS_PER_DIVIDED_FRAGMENT})."
        )

    fragment_counts = (
        fragment_df.groupby("tablet_id")
        .size()
        .rename("num_valid_fragments_in_tablet")
    )

    fragment_df = fragment_df.merge(
        fragment_counts,
        left_on="tablet_id",
        right_index=True,
        how="left",
    )

    if not KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS:
        keep_tablets = fragment_counts[
            fragment_counts >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET
        ].index
        fragment_df = fragment_df[
            fragment_df["tablet_id"].isin(keep_tablets)
        ].copy()

    valid_fragment_uids = set(fragment_df["fragment_uid"])
    assigned = assigned[
        assigned["fragment_uid"].isin(valid_fragment_uids)
    ].copy()

    # Join the original annotation rows using the stable source-row index.
    annotation_columns = [
        "source_row_index",
        "crop_path",
        "tablet_id",
        "period",
        "sign_name",
        "x",
        "y",
        "x2",
        "y2",
        "center_x",
        "center_y",
    ]

    annotation_lookup = annotation_df[annotation_columns].copy()

    assigned = assigned.drop(
        columns=[
            column
            for column in [
                "crop_path",
                "tablet_id",
                "period",
                "sign_name",
            ]
            if column in assigned.columns
        ]
    )

    assigned_sign_df = assigned.merge(
        annotation_lookup,
        on="source_row_index",
        how="inner",
    )

    assigned_sign_df = assigned_sign_df.sort_values(
        ["tablet_id", "fragment_uid", "source_row_index"]
    ).reset_index(drop=True)

    fragment_period = (
        assigned_sign_df.groupby("fragment_uid")["period"]
        .agg(lambda values: values.mode().iloc[0])
        .rename("period")
        .reset_index()
    )

    fragment_sign_names = (
        assigned_sign_df.groupby("fragment_uid")["sign_name"]
        .agg(lambda values: ";".join(map(str, values)))
        .rename("sign_names")
        .reset_index()
    )

    source_indices = (
        assigned_sign_df.groupby("fragment_uid")["source_row_index"]
        .agg(lambda values: ";".join(map(str, values.astype(int))))
        .rename("source_row_indices")
        .reset_index()
    )

    fragment_df = fragment_df.merge(
        fragment_period,
        on="fragment_uid",
        how="left",
    )
    fragment_df = fragment_df.merge(
        fragment_sign_names,
        on="fragment_uid",
        how="left",
    )
    fragment_df = fragment_df.merge(
        source_indices,
        on="fragment_uid",
        how="left",
    )

    fragment_df = fragment_df.sort_values(
        ["tablet_id", "fragment_id"]
    ).reset_index(drop=True)

    fragment_df.to_csv(
        OUTPUT_DIR / "valid_divided_fragment_metadata.csv",
        index=False,
    )
    assigned_sign_df.to_csv(
        OUTPUT_DIR / "assigned_signs_used_for_embeddings.csv",
        index=False,
    )

    print("\nDIVIDED-FRAGMENT DATASET")
    print("------------------------")
    print("Assigned signs used:", f"{len(assigned_sign_df):,}")
    print("Valid divided fragments:", f"{len(fragment_df):,}")
    print(
        "Parent tablets represented:",
        f"{fragment_df['tablet_id'].nunique():,}",
    )
    print(
        "Tablets with at least two valid fragments:",
        f"{fragment_df.loc[fragment_df['num_valid_fragments_in_tablet'] >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET, 'tablet_id'].nunique():,}",
    )

    return fragment_df, assigned_sign_df


# ============================================================
# SIGN-CROP DATASET AND CONVNEXT EMBEDDING EXTRACTION
# ============================================================

test_transform = transforms.Compose(
    [
        transforms.Resize((232, 232)),
        transforms.CenterCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)


class AssignedSignCropDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        crop_path = Path(str(row["crop_path"]))

        if not crop_path.is_file():
            # Try to recover a crop path copied from another Windows machine.
            filename = PureWindowsPath(str(row["crop_path"])).name
            recovered_candidates = list(DATA_ROOT.rglob(filename))
            if recovered_candidates:
                crop_path = recovered_candidates[0]

        try:
            with Image.open(crop_path) as image:
                image = image.convert("RGB")
                image = self.transform(image)
        except Exception as error:
            raise RuntimeError(
                f"Could not open annotated sign crop at assigned row "
                f"{index}: {crop_path}"
            ) from error

        return image, index


def normalize_state_dict_keys(
    state_dict: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    cleaned: Dict[str, torch.Tensor] = {}

    removable_prefixes = (
        "module.",
        "model.",
        "network.",
        "backbone.",
    )

    for original_key, value in state_dict.items():
        key = original_key
        changed = True
        while changed:
            changed = False
            for prefix in removable_prefixes:
                if key.startswith(prefix):
                    key = key[len(prefix):]
                    changed = True
        cleaned[key] = value

    return cleaned


def extract_state_dict(checkpoint: object) -> Dict[str, torch.Tensor]:
    if not isinstance(checkpoint, dict):
        return checkpoint

    for key in (
        "model_state_dict",
        "state_dict",
        "model",
        "network_state_dict",
    ):
        candidate = checkpoint.get(key)
        if isinstance(candidate, dict):
            return candidate

    if all(isinstance(value, torch.Tensor) for value in checkpoint.values()):
        return checkpoint

    raise ValueError(
        "Could not identify a model state dictionary in the checkpoint."
    )


def resolve_checkpoint(
    checkpoint_candidates: Sequence[str],
    architecture: str,
) -> Path:
    """Resolve a ConvNeXt checkpoint using exact names and keyword fallback."""
    for checkpoint_name in checkpoint_candidates:
        direct_candidates = [
            Path(checkpoint_name),
            Path.cwd() / checkpoint_name,
            MODEL_SEARCH_ROOT / checkpoint_name,
        ]

        for candidate in direct_candidates:
            if candidate.is_file():
                return candidate.resolve()

        recursive_matches = sorted(
            path.resolve()
            for path in MODEL_SEARCH_ROOT.rglob(checkpoint_name)
            if path.is_file()
        )

        if recursive_matches:
            if len(recursive_matches) > 1:
                warnings.warn(
                    f"Multiple checkpoints named {checkpoint_name} were found; "
                    f"using {recursive_matches[0]}"
                )
            return recursive_matches[0]

    keywords = ("convnext", "tablet", "holdout")
    fallback_matches = sorted(
        path.resolve()
        for path in MODEL_SEARCH_ROOT.rglob("*.pth")
        if path.is_file()
        and all(keyword in path.name.lower() for keyword in keywords)
    )

    if len(fallback_matches) == 1:
        warnings.warn(
            "No exact checkpoint filename matched. "
            f"Using keyword match: {fallback_matches[0]}"
        )
        return fallback_matches[0]

    if len(fallback_matches) > 1:
        raise FileNotFoundError(
            "Multiple possible ConvNeXt checkpoints were found:\n"
            + "\n".join(f"  {path}" for path in fallback_matches)
            + "\nPlace the intended filename first in "
              "MODEL_SPECS['convnext_base_tablet_holdout']"
              "['checkpoint_candidates']."
        )

    raise FileNotFoundError(
        f"No checkpoint found for architecture {architecture}.\n"
        f"Candidate filenames: {list(checkpoint_candidates)}\n"
        f"Searched recursively under: {MODEL_SEARCH_ROOT}"
    )


def load_convnext_embedding_model(
    architecture: str,
    checkpoint_path: Path,
    device: str,
) -> Tuple[nn.Module, int]:
    """Load a trained ConvNeXt Base checkpoint and expose penultimate embeddings."""
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )
    state_dict = normalize_state_dict_keys(
        extract_state_dict(checkpoint)
    )

    if architecture != "convnext_base":
        raise ValueError(
            f"Unsupported ConvNeXt architecture: {architecture}"
        )

    classifier_weight_key = next(
        (
            key
            for key in state_dict
            if key.endswith("classifier.2.weight")
        ),
        None,
    )

    if classifier_weight_key is None:
        classifier_keys = [
            key for key in state_dict if "classifier" in key
        ]
        raise ValueError(
            "Could not find classifier.2.weight in the ConvNeXt checkpoint: "
            f"{checkpoint_path}\n"
            f"Classifier-related keys: {classifier_keys[-20:]}"
        )

    num_classes = int(
        state_dict[classifier_weight_key].shape[0]
    )

    model = models.convnext_base(weights=None)

    if (
        not isinstance(model.classifier, nn.Sequential)
        or len(model.classifier) < 3
        or not isinstance(model.classifier[2], nn.Linear)
    ):
        raise TypeError(
            "Unexpected torchvision ConvNeXt classifier structure. "
            "Expected the final Linear layer at model.classifier[2]."
        )

    embedding_dim = int(model.classifier[2].in_features)

    model.classifier[2] = nn.Linear(
        embedding_dim,
        num_classes,
    )

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as error:
        raise RuntimeError(
            "The checkpoint does not match torchvision.models.convnext_base. "
            "Confirm that it was trained with ConvNeXt Base and the same "
            "torchvision model definition.\n"
            f"Checkpoint: {checkpoint_path}"
        ) from error

    # Preserve ConvNeXt's final normalization and flattening while removing
    # only the classification layer. The output is the penultimate embedding.
    model.classifier[2] = nn.Identity()

    model = model.to(device)
    model.eval()

    print("Architecture:", architecture)
    print("Checkpoint:", checkpoint_path)
    print("Detected classes:", num_classes)
    print("Embedding dimension:", embedding_dim)

    return model, embedding_dim


def extract_sign_embeddings(
    model: nn.Module,
    embedding_dim: int,
    data_loader: DataLoader,
    total_samples: int,
    device: str,
) -> np.ndarray:
    embeddings = np.zeros(
        (total_samples, embedding_dim),
        dtype=np.float32,
    )

    with torch.inference_mode():
        for images, indices in tqdm(
            data_loader,
            desc="Extracting annotated sign embeddings",
        ):
            images = images.to(
                device,
                non_blocking=PIN_MEMORY,
            )

            batch_embeddings = model(images)
            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

            embeddings[indices.numpy()] = (
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

    return embeddings


# ============================================================
# AVERAGE SIGN EMBEDDINGS WITHIN EACH DIVIDED FRAGMENT
# ============================================================

def create_divided_fragment_embeddings(
    fragment_metadata_df: pd.DataFrame,
    assigned_sign_df: pd.DataFrame,
    sign_embeddings: np.ndarray,
) -> Tuple[pd.DataFrame, np.ndarray]:

    required_fragment_columns = {
        "fragment_uid",
        "tablet_id",
        "fragment_id",
    }

    required_sign_columns = {
        "fragment_uid",
    }

    missing_fragment_columns = (
        required_fragment_columns
        - set(fragment_metadata_df.columns)
    )

    missing_sign_columns = (
        required_sign_columns
        - set(assigned_sign_df.columns)
    )

    if missing_fragment_columns:
        raise ValueError(
            "fragment_metadata_df is missing columns: "
            f"{sorted(missing_fragment_columns)}"
        )

    if missing_sign_columns:
        raise ValueError(
            "assigned_sign_df is missing columns: "
            f"{sorted(missing_sign_columns)}"
        )

    fragment_metadata_df = fragment_metadata_df.copy()
    assigned_sign_df = assigned_sign_df.copy()

    fragment_metadata_df["fragment_uid"] = (
        fragment_metadata_df["fragment_uid"]
        .astype(str)
        .str.strip()
    )

    assigned_sign_df["fragment_uid"] = (
        assigned_sign_df["fragment_uid"]
        .astype(str)
        .str.strip()
    )

    metadata_lookup = fragment_metadata_df.set_index(
        "fragment_uid",
        drop=True,
    )

    fragment_rows: List[Dict[str, object]] = []
    fragment_embeddings: List[np.ndarray] = []

    for fragment_uid, group in assigned_sign_df.groupby(
        "fragment_uid",
        sort=False,
    ):
        fragment_uid = str(fragment_uid).strip()

        if fragment_uid not in metadata_lookup.index:
            continue

        sign_indices = group.index.to_numpy(dtype=np.int64)

        if sign_indices.size == 0:
            continue

        if sign_indices.max() >= len(sign_embeddings):
            raise IndexError(
                f"Sign index {sign_indices.max()} exceeds embedding "
                f"array length {len(sign_embeddings)}."
            )

        mean_embedding = sign_embeddings[sign_indices].mean(axis=0)
        norm = float(np.linalg.norm(mean_embedding))

        if not np.isfinite(norm) or norm <= 1e-8:
            continue

        mean_embedding = (
            mean_embedding / norm
        ).astype(np.float32)

        metadata = metadata_lookup.loc[fragment_uid]

        if isinstance(metadata, pd.DataFrame):
            metadata = metadata.iloc[0]

        metadata_record = metadata.to_dict()

        # Explicitly restore the index column.
        metadata_record["fragment_uid"] = fragment_uid

        fragment_rows.append(metadata_record)
        fragment_embeddings.append(mean_embedding)

    if not fragment_embeddings:
        raise ValueError(
            "No divided-fragment embeddings were created."
        )

    output_df = pd.DataFrame(
        fragment_rows
    ).reset_index(drop=True)

    output_embeddings = np.vstack(
        fragment_embeddings
    ).astype(np.float32)

    if "fragment_uid" not in output_df.columns:
        raise ValueError(
            "fragment_uid was lost while creating fragment metadata."
        )

    if len(output_df) != len(output_embeddings):
        raise ValueError(
            "Fragment metadata and embedding counts do not match: "
            f"{len(output_df)} metadata rows versus "
            f"{len(output_embeddings)} embeddings."
        )

    # Recompute valid-fragment counts after removing invalid embeddings.
    valid_counts = (
        output_df.groupby("tablet_id")
        .size()
        .rename("num_valid_fragments_in_tablet")
    )

    output_df = (
        output_df.drop(
            columns=["num_valid_fragments_in_tablet"],
            errors="ignore",
        )
        .merge(
            valid_counts,
            left_on="tablet_id",
            right_index=True,
            how="left",
        )
    )

    if not KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS:

        keep_tablets = valid_counts[
            valid_counts
            >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET
        ].index

        keep_mask = (
            output_df["tablet_id"]
            .isin(keep_tablets)
            .to_numpy()
        )

        output_df = (
            output_df.loc[keep_mask]
            .reset_index(drop=True)
        )

        output_embeddings = output_embeddings[keep_mask]

    if SHUFFLE_FRAGMENT_GALLERY:

        permutation = (
            np.random.default_rng(RANDOM_SEED)
            .permutation(len(output_df))
        )

        output_df = (
            output_df.iloc[permutation]
            .reset_index(drop=True)
        )

        output_embeddings = output_embeddings[permutation]

    print(
        "Fragment metadata columns:",
        output_df.columns.tolist(),
    )

    return output_df, output_embeddings


# ============================================================
# RETRIEVAL EVALUATION
# ============================================================

def evaluate_fragment_retrieval(
    fragment_df: pd.DataFrame,
    fragment_embeddings: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    similarity_matrix = cosine_similarity(fragment_embeddings)

    eligible_query_indices = fragment_df.index[
        fragment_df["num_valid_fragments_in_tablet"]
        >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET
    ].tolist()

    rows: List[Dict[str, object]] = []

    for query_index in tqdm(
        eligible_query_indices,
        desc="Evaluating divided-fragment retrieval",
    ):
        query_row = fragment_df.loc[query_index]
        query_tablet = query_row["tablet_id"]

        similarities = similarity_matrix[query_index].copy()
        similarities[query_index] = -np.inf
        ranked_indices = np.argsort(similarities)[::-1]

        positive_indices = set(
            fragment_df.index[
                (fragment_df["tablet_id"] == query_tablet)
                & (fragment_df.index != query_index)
            ].tolist()
        )

        if not positive_indices:
            continue

        first_positive_rank: Optional[int] = None
        first_positive_index: Optional[int] = None

        for rank, candidate_index in enumerate(ranked_indices, start=1):
            if int(candidate_index) in positive_indices:
                first_positive_rank = rank
                first_positive_index = int(candidate_index)
                break

        if first_positive_index is None or first_positive_rank is None:
            continue

        positive_row = fragment_df.loc[first_positive_index]

        rows.append(
            {
                "query_fragment": query_row["fragment_uid"],
                "query_tablet": query_tablet,
                "query_period": query_row["period"],
                "query_num_signs": int(query_row["num_signs"]),
                "num_positive_fragments": int(len(positive_indices)),
                "first_positive_rank": int(first_positive_rank),
                "first_positive_fragment": positive_row["fragment_uid"],
                "first_positive_num_signs": int(positive_row["num_signs"]),
                "first_positive_similarity": float(
                    similarities[first_positive_index]
                ),
                "reciprocal_rank": 1.0 / first_positive_rank,
                "hit_at_1": int(first_positive_rank <= 1),
                "hit_at_5": int(first_positive_rank <= 5),
                "hit_at_10": int(first_positive_rank <= 10),
            }
        )

    retrieval_df = pd.DataFrame(rows)
    if retrieval_df.empty:
        raise ValueError(
            "No eligible retrieval queries were found. At least one parent "
            "tablet must produce two valid divided fragments."
        )

    metrics = {
        "fragment_extraction_method": "divide_tablet_photo",
        "min_signs_per_divided_fragment": (
            MIN_SIGNS_PER_DIVIDED_FRAGMENT
        ),
        "num_query_fragments": int(len(retrieval_df)),
        "num_gallery_fragments": int(len(fragment_df)),
        "num_parent_tablets_in_gallery": int(
            fragment_df["tablet_id"].nunique()
        ),
        "num_parent_tablets_with_multiple_fragments": int(
            fragment_df.loc[
                fragment_df["num_valid_fragments_in_tablet"]
                >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET,
                "tablet_id",
            ].nunique()
        ),
        "Recall@1": float(retrieval_df["hit_at_1"].mean()),
        "Recall@5": float(retrieval_df["hit_at_5"].mean()),
        "Recall@10": float(retrieval_df["hit_at_10"].mean()),
        "MRR": float(retrieval_df["reciprocal_rank"].mean()),
        "mean_first_positive_rank": float(
            retrieval_df["first_positive_rank"].mean()
        ),
        "median_first_positive_rank": float(
            retrieval_df["first_positive_rank"].median()
        ),
    }

    return retrieval_df, pd.DataFrame([metrics]), similarity_matrix


def save_topk_retrieval(
    fragment_df: pd.DataFrame,
    similarity_matrix: np.ndarray,
    output_path: Path,
    top_k: int,
) -> pd.DataFrame:
    topk_rows: List[Dict[str, object]] = []

    eligible_query_indices = fragment_df.index[
        fragment_df["num_valid_fragments_in_tablet"]
        >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET
    ].tolist()

    effective_top_k = min(top_k, len(fragment_df) - 1)

    for query_index in tqdm(
        eligible_query_indices,
        desc=f"Saving top-{effective_top_k} divided-fragment retrieval",
    ):
        query_row = fragment_df.loc[query_index]
        similarities = similarity_matrix[query_index].copy()
        similarities[query_index] = -np.inf
        ranked_indices = np.argsort(similarities)[::-1][:effective_top_k]

        for rank, candidate_index in enumerate(ranked_indices, start=1):
            candidate_row = fragment_df.loc[int(candidate_index)]

            topk_rows.append(
                {
                    "query_fragment": query_row["fragment_uid"],
                    "query_tablet": query_row["tablet_id"],
                    "query_period": query_row["period"],
                    "query_num_signs": int(query_row["num_signs"]),
                    "query_fragment_path": query_row["fragment_path"],
                    "rank": rank,
                    "candidate_fragment": candidate_row["fragment_uid"],
                    "candidate_tablet": candidate_row["tablet_id"],
                    "candidate_period": candidate_row["period"],
                    "candidate_num_signs": int(candidate_row["num_signs"]),
                    "candidate_fragment_path": candidate_row["fragment_path"],
                    "same_parent_tablet": bool(
                        query_row["tablet_id"]
                        == candidate_row["tablet_id"]
                    ),
                    "similarity": float(similarities[int(candidate_index)]),
                }
            )

    topk_df = pd.DataFrame(topk_rows)
    topk_df.to_csv(output_path, index=False)
    return topk_df


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    """Resume retrieval from saved fragment metadata without re-dividing tablets."""
    annotation_df = load_annotation_dataframe(CSV_PATH)

    valid_fragment_csv = (
        FRAGMENT_METADATA_DIR / "valid_divided_fragment_metadata.csv"
    )
    assigned_signs_csv = (
        FRAGMENT_METADATA_DIR / "assigned_signs_used_for_embeddings.csv"
    )

    localization_csv = (
        FRAGMENT_METADATA_DIR / "all_divided_fragment_localizations.csv"
    )
    assignments_csv = (
        FRAGMENT_METADATA_DIR / "all_annotation_to_fragment_assignments.csv"
    )
    tablet_status_csv = (
        FRAGMENT_METADATA_DIR / "tablet_division_processing_status.csv"
    )

    print("\nRESUME FILE CHECK")
    print("-----------------")
    for path in (
        valid_fragment_csv,
        assigned_signs_csv,
        localization_csv,
        assignments_csv,
        tablet_status_csv,
    ):
        print(f"{path.name}: {path.is_file()} | {path}")

    resume_source = ""

    # Fastest and safest resume point: final filtered fragment/sign tables.
    if valid_fragment_csv.is_file() and assigned_signs_csv.is_file():
        resume_source = "valid_fragment_tables"
        print("\nFAST RESUME SELECTED")
        print("Previously extracted spatial fragments will NOT be divided or saved again.")

        fragment_metadata_df = pd.read_csv(
            valid_fragment_csv,
            low_memory=False,
        )
        assigned_sign_df = pd.read_csv(
            assigned_signs_csv,
            low_memory=False,
        )

    # Earlier resume point: localization and annotation-assignment tables.
    elif (
        localization_csv.is_file()
        and assignments_csv.is_file()
        and tablet_status_csv.is_file()
    ):
        resume_source = "localization_and_assignment_tables"
        print("\nMETADATA RESUME SELECTED")
        print("Previously extracted spatial fragments will NOT be divided or saved again.")

        localization_df = pd.read_csv(
            localization_csv,
            low_memory=False,
        )
        assignments_df = pd.read_csv(
            assignments_csv,
            low_memory=False,
        )
        tablet_status_df = pd.read_csv(
            tablet_status_csv,
            low_memory=False,
        )

        required_localization_columns = {
            "tablet_id",
            "fragment_id",
            "fragment_uid",
            "fragment_path",
            "localization_status",
        }
        required_assignment_columns = {
            "source_row_index",
            "tablet_id",
            "fragment_id",
            "fragment_uid",
            "assignment_status",
        }

        missing_localization = (
            required_localization_columns - set(localization_df.columns)
        )
        missing_assignments = (
            required_assignment_columns - set(assignments_df.columns)
        )

        if missing_localization:
            raise ValueError(
                "Saved localization CSV is missing columns: "
                f"{sorted(missing_localization)}"
            )
        if missing_assignments:
            raise ValueError(
                "Saved assignment CSV is missing columns: "
                f"{sorted(missing_assignments)}"
            )

        print("\nFRAGMENT LOCALIZATION SUMMARY")
        print("------------------------------")
        print(
            localization_df["localization_status"]
            .value_counts(dropna=False)
            .to_string()
        )

        print("\nANNOTATION ASSIGNMENT SUMMARY")
        print("-----------------------------")
        print(
            assignments_df["assignment_status"]
            .value_counts(dropna=False)
            .to_string()
        )

        fragment_metadata_df, assigned_sign_df = build_valid_fragment_dataset(
            annotation_df=annotation_df,
            localization_df=localization_df,
            assignments_df=assignments_df,
        )

    else:
        raise FileNotFoundError(
            "No reusable fragment metadata files were found.\n"
            "The program stopped intentionally so that tablet division is not "
            "run again.\n\n"
            "Expected either:\n"
            f"  {valid_fragment_csv}\n"
            f"  {assigned_signs_csv}\n\n"
            "or all of:\n"
            f"  {localization_csv}\n"
            f"  {assignments_csv}\n"
            f"  {tablet_status_csv}"
        )

    required_fragment_columns = {
        "fragment_uid",
        "tablet_id",
        "fragment_id",
        "num_signs",
    }
    required_sign_columns = {
        "fragment_uid",
        "tablet_id",
        "crop_path",
        "source_row_index",
    }

    missing_fragment_columns = (
        required_fragment_columns - set(fragment_metadata_df.columns)
    )
    missing_sign_columns = required_sign_columns - set(assigned_sign_df.columns)

    if missing_fragment_columns:
        raise ValueError(
            "Valid-fragment metadata is missing columns: "
            f"{sorted(missing_fragment_columns)}"
        )
    if missing_sign_columns:
        raise ValueError(
            "Assigned-sign metadata is missing columns: "
            f"{sorted(missing_sign_columns)}"
        )

    # Row positions must match positions in the sign-embedding array.
    fragment_metadata_df = fragment_metadata_df.reset_index(drop=True)
    assigned_sign_df = assigned_sign_df.reset_index(drop=True)

    print("\nDATASET READY")
    print("-------------")
    print("Resume source:", resume_source)
    print("Valid divided fragments:", f"{len(fragment_metadata_df):,}")
    print("Assigned signs:", f"{len(assigned_sign_df):,}")
    print(
        "Parent tablets:",
        f"{fragment_metadata_df['tablet_id'].nunique():,}",
    )

    if PREVIEW_ONLY:
        print(
            "\nPREVIEW_ONLY=True. Saved metadata was loaded, but embedding "
            "extraction and retrieval were skipped."
        )
        return

    run_config = {
        "csv_path": str(CSV_PATH),
        "tablet_image_dir": str(TABLET_IMAGE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "fragment_metadata_dir": str(FRAGMENT_METADATA_DIR),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "resume_mode": True,
        "resume_source": resume_source,
        "spatial_fragment_extraction_skipped": True,
        "use_tablet_division": USE_TABLET_DIVISION,
        "min_fragment_width": MIN_FRAGMENT_WIDTH,
        "min_fragment_height": MIN_FRAGMENT_HEIGHT,
        "min_fragment_localization_score": MIN_FRAGMENT_LOCALIZATION_SCORE,
        "use_orb_fallback": USE_ORB_FALLBACK,
        "min_annotation_overlap_ratio": MIN_ANNOTATION_OVERLAP_RATIO,
        "min_signs_per_divided_fragment": MIN_SIGNS_PER_DIVIDED_FRAGMENT,
        "keep_single_fragment_tablets_as_distractors": (
            KEEP_SINGLE_FRAGMENT_TABLETS_AS_DISTRACTORS
        ),
        "top_k_to_save": TOP_K_TO_SAVE,
        "shuffle_fragment_gallery": SHUFFLE_FRAGMENT_GALLERY,
        "random_seed": RANDOM_SEED,
    }

    with open(
        OUTPUT_DIR / "run_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(run_config, file, indent=2)

    dataset = AssignedSignCropDataset(
        assigned_sign_df,
        test_transform,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    all_metrics: List[pd.DataFrame] = []

    for model_name, model_spec in MODEL_SPECS.items():
        print("\n" + "=" * 90)
        print("PROCESSING MODEL:", model_name)
        print("=" * 90)

        architecture = model_spec["architecture"]
        checkpoint_path = resolve_checkpoint(
            checkpoint_candidates=model_spec["checkpoint_candidates"],
            architecture=architecture,
        )
        model, embedding_dim = load_convnext_embedding_model(
            architecture=architecture,
            checkpoint_path=checkpoint_path,
            device=DEVICE,
        )

        sign_embedding_path = (
            OUTPUT_DIR
            / f"{model_name}_assigned_sign_embeddings.npy"
        )

        if sign_embedding_path.is_file():

            print(
                "Loading previously saved sign embeddings:",
                sign_embedding_path,
            )

            sign_embeddings = np.load(sign_embedding_path)

            expected_shape = (
                len(assigned_sign_df),
                embedding_dim,
            )

            if sign_embeddings.shape != expected_shape:
                print(
                    "Saved sign embeddings have an incompatible shape:",
                    sign_embeddings.shape,
                )
                print(
                    "Expected:",
                    expected_shape,
                )
                print("Re-extracting sign embeddings.")

                sign_embeddings = extract_sign_embeddings(
                    model=model,
                    embedding_dim=embedding_dim,
                    data_loader=loader,
                    total_samples=len(assigned_sign_df),
                    device=DEVICE,
                )

                np.save(
                    sign_embedding_path,
                    sign_embeddings,
                )

        else:
            sign_embeddings = extract_sign_embeddings(
                model=model,
                embedding_dim=embedding_dim,
                data_loader=loader,
                total_samples=len(assigned_sign_df),
                device=DEVICE,
            )

            np.save(
                sign_embedding_path,
                sign_embeddings,
            )

        fragment_df, fragment_embeddings = create_divided_fragment_embeddings(
            fragment_metadata_df=fragment_metadata_df,
            assigned_sign_df=assigned_sign_df,
            sign_embeddings=sign_embeddings,
        )

        print("Divided fragments:", f"{len(fragment_df):,}")
        print(
            "Parent tablets represented:",
            f"{fragment_df['tablet_id'].nunique():,}",
        )
        print(
            "Parent tablets with at least two fragments:",
            f"{fragment_df.loc[fragment_df['num_valid_fragments_in_tablet'] >= MIN_VALID_FRAGMENTS_PER_QUERY_TABLET, 'tablet_id'].nunique():,}",
        )
        print("Fragment embedding shape:", fragment_embeddings.shape)

        fragment_df.to_csv(
            OUTPUT_DIR / f"{model_name}_divided_fragment_metadata.csv",
            index=False,
        )
        np.save(
            OUTPUT_DIR / f"{model_name}_divided_fragment_embeddings.npy",
            fragment_embeddings,
        )

        retrieval_df, metrics_df, similarity_matrix = evaluate_fragment_retrieval(
            fragment_df=fragment_df,
            fragment_embeddings=fragment_embeddings,
        )

        metrics_df.insert(0, "model", model_name)

        retrieval_df.to_csv(
            OUTPUT_DIR / f"{model_name}_divided_fragment_retrieval_results.csv",
            index=False,
        )
        metrics_df.to_csv(
            OUTPUT_DIR / f"{model_name}_divided_fragment_retrieval_metrics.csv",
            index=False,
        )

        save_topk_retrieval(
            fragment_df=fragment_df,
            similarity_matrix=similarity_matrix,
            output_path=(
                OUTPUT_DIR
                / f"{model_name}_divided_fragment_top{TOP_K_TO_SAVE}.csv"
            ),
            top_k=TOP_K_TO_SAVE,
        )

        all_metrics.append(metrics_df)

        print("\nDIVIDED-FRAGMENT RETRIEVAL METRICS")
        print(metrics_df.to_string(index=False))

        del model
        del sign_embeddings
        del fragment_embeddings
        del similarity_matrix
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not all_metrics:
        raise ValueError("No model metrics were generated.")

    combined_metrics_df = pd.concat(
        all_metrics,
        ignore_index=True,
    )
    combined_metrics_path = (
        OUTPUT_DIR / "all_convnext_divided_fragment_retrieval_metrics.csv"
    )
    combined_metrics_df.to_csv(combined_metrics_path, index=False)

    print("\n" + "=" * 90)
    print("COMBINED CONVNEXT DIVIDED-FRAGMENT RETRIEVAL METRICS")
    print("=" * 90)
    print(combined_metrics_df.to_string(index=False))
    print("\nSaved all results to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()